> **Repository version.** This notebook was cleaned from the frozen manuscript workflow:
> outputs were removed and local absolute paths were replaced by the repository `PROJECT_DIR`.
> The statistical logic was not intentionally changed.


In [ ]:
# Repository path setup
from pathlib import Path
import os

_here = Path.cwd().resolve()
REPO_ROOT = _here.parent if _here.name == "notebooks" else _here
PROJECT_DIR = Path(
    os.environ.get("CGN_PROJECT_DIR", REPO_ROOT / "workspace")
).expanduser().resolve()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "figures").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "reproduction" / "results").mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Working data/results directory:", PROJECT_DIR)


In [ ]:
# ============================================================
# FIGURE 5 — PART 1
# MEMORY-SAFE ROI × MAC/FIB EXPRESSION EXTRACTION
#
# Output:
# one row = one ROI × one cell type
# columns = metadata + 480 mean-count genes
#
# ============================================================

from pathlib import Path

import h5py
import anndata as ad
import pandas as pd
import numpy as np
import scipy.sparse as sp


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

big_path = (
    base /
    "GSE294965_processed_data.h5ad"
)

roi_path = (
    base /
    "roi_782_PC1_primary.h5ad"
)

cell_meta_path = (
    base /
    "roi782_cell_metadata.pkl"
)

output_path = (
    base /
    "figure5_MAC_FIB_roi_mean_counts.csv"
)


# ============================================================
# 1. Load small files
# ============================================================

roi_ad = ad.read_h5ad(
    roi_path
)

cell_meta = pd.read_pickle(
    cell_meta_path
)

genes = (
    roi_ad.var_names
    .astype(str)
    .to_numpy()
)

print(
    "ROI object:",
    roi_ad.shape
)

print(
    "Cell metadata:",
    cell_meta.shape
)

print(
    "Genes:",
    len(genes)
)


# ============================================================
# 2. Use EXACTLY the same 3-disease common PC1 support
#    as Figure 3/4
# ============================================================

pc_ranges = (
    roi_ad.obs[
        roi_ad.obs["Disease"].isin(
            ["ANCA", "SLE", "GBM"]
        )
    ]
    .groupby(
        "Disease",
        observed=True
    )["PC1_crescent"]
    .agg(
        ["min", "max"]
    )
)

pc_low = (
    pc_ranges["min"].max()
)

pc_high = (
    pc_ranges["max"].min()
)

print(
    "\nCommon PC1 support:",
    pc_low,
    "to",
    pc_high
)


# ============================================================
# 3. LN + anti-GBM ROIs within common support
# ============================================================

analysis_meta = roi_ad.obs[
    roi_ad.obs["Disease"].isin(
        ["SLE", "GBM"]
    )
    &
    roi_ad.obs["PC1_crescent"].between(
        pc_low,
        pc_high
    )
].copy()

analysis_meta.index = (
    analysis_meta.index.astype(str)
)

analysis_roi_ids = set(
    analysis_meta.index
)

print(
    "\nROIs:"
)

print(
    analysis_meta["Disease"]
    .value_counts()
)

print(
    "\nPatients:"
)

print(
    analysis_meta
    .groupby(
        "Disease",
        observed=True
    )["Patient_Sample_ID"]
    .nunique()
)


# ============================================================
# 4. Count MAC/FIB cells per ROI
# ============================================================

cm = cell_meta.copy()

cm["roi_id"] = (
    cm["roi_id"]
    .astype(str)
)

cm["celltype_l1"] = (
    cm["celltype_l1"]
    .astype(str)
)

cm = cm[
    cm["roi_id"].isin(
        analysis_roi_ids
    )
    &
    cm["celltype_l1"].isin(
        ["MAC", "FIB"]
    )
].copy()


roi_ct_counts = (
    cm.groupby(
        [
            "roi_id",
            "celltype_l1"
        ],
        observed=True
    )
    .size()
    .rename(
        "n_cells"
    )
    .reset_index()
)


# ============================================================
# 5. Coverage audit
# ============================================================

audit_rows = []

for threshold in [
    3,
    5,
    10,
    20
]:

    for ct in [
        "MAC",
        "FIB"
    ]:

        eligible_ids = (
            roi_ct_counts.loc[
                (
                    roi_ct_counts[
                        "celltype_l1"
                    ]
                    == ct
                )
                &
                (
                    roi_ct_counts[
                        "n_cells"
                    ]
                    >= threshold
                ),
                "roi_id"
            ]
            .astype(str)
        )

        temp = (
            analysis_meta.loc[
                analysis_meta.index.isin(
                    eligible_ids
                )
            ]
        )

        audit_rows.append({
            "min_cells":
                threshold,

            "celltype":
                ct,

            "n_ROI":
                len(temp),

            "n_patients":
                temp[
                    "Patient_Sample_ID"
                ].nunique(),

            "n_LN_patients":
                temp.loc[
                    temp["Disease"]
                    == "SLE",
                    "Patient_Sample_ID"
                ].nunique(),

            "n_GBM_patients":
                temp.loc[
                    temp["Disease"]
                    == "GBM",
                    "Patient_Sample_ID"
                ].nunique()
        })


coverage_audit = pd.DataFrame(
    audit_rows
)

print(
    "\n=============================="
)

print(
    "ROI × CELLTYPE COVERAGE AUDIT"
)

print(
    "=============================="
)

display(
    coverage_audit
)


# ============================================================
# 6. Pre-specified molecular threshold
#
# Require >=10 cells of that cell type per ROI.
#
# This is NOT chosen based on significance.
# ============================================================

MIN_CELLS = 10

eligible = roi_ct_counts[
    roi_ct_counts["n_cells"]
    >= MIN_CELLS
].copy()

eligible = eligible.sort_values(
    [
        "celltype_l1",
        "roi_id"
    ]
).reset_index(
    drop=True
)

print(
    "\nEligible ROI × celltype units:",
    len(eligible)
)

print(
    eligible[
        "celltype_l1"
    ].value_counts()
)


# ============================================================
# 7. Attach metadata
# ============================================================

eligible["Disease"] = (
    eligible["roi_id"]
    .map(
        analysis_meta[
            "Disease"
        ]
    )
)

eligible["Patient"] = (
    eligible["roi_id"]
    .map(
        analysis_meta[
            "Patient_Sample_ID"
        ].astype(str)
    )
)

eligible["PC1"] = (
    eligible["roi_id"]
    .map(
        analysis_meta[
            "PC1_crescent"
        ]
    )
)


print(
    "\nPatient coverage at MIN_CELLS=10:"
)

print(
    eligible
    .groupby(
        [
            "celltype_l1",
            "Disease"
        ],
        observed=True
    )["Patient"]
    .nunique()
)


# ============================================================
# 8. Build group lookup
# ============================================================

group_keys = list(
    zip(
        eligible[
            "roi_id"
        ].astype(str),

        eligible[
            "celltype_l1"
        ].astype(str)
    )
)

group_map = {
    key: i
    for i, key
    in enumerate(
        group_keys
    )
}

n_groups = len(
    group_keys
)

eligible_roi_ids = set(
    eligible[
        "roi_id"
    ].astype(str)
)


# ============================================================
# 9. H5AD helper functions
# ============================================================

def decode_array(arr):

    arr = np.asarray(
        arr
    )

    if arr.dtype.kind in {
        "S",
        "O",
        "U"
    }:

        return np.array(
            [
                (
                    x.decode(
                        "utf-8"
                    )
                    if isinstance(
                        x,
                        (
                            bytes,
                            np.bytes_
                        )
                    )
                    else str(x)
                )
                for x in arr
            ],
            dtype=object
        )

    return arr


def read_obs_slice(
    obs_group,
    name,
    row_slice
):

    node = obs_group[
        name
    ]

    # normal dataset
    if isinstance(
        node,
        h5py.Dataset
    ):

        return decode_array(
            node[
                row_slice
            ]
        )

    # categorical
    if (
        "codes" in node
        and
        "categories" in node
    ):

        codes = np.asarray(
            node[
                "codes"
            ][
                row_slice
            ]
        )

        categories = decode_array(
            node[
                "categories"
            ][:]
        )

        out = np.empty(
            len(codes),
            dtype=object
        )

        out[:] = None

        good = (
            codes >= 0
        )

        out[
            good
        ] = categories[
            codes[
                good
            ]
        ]

        return out

    # nullable
    if (
        "values" in node
        and
        "mask" in node
    ):

        values = decode_array(
            node[
                "values"
            ][
                row_slice
            ]
        )

        mask = np.asarray(
            node[
                "mask"
            ][
                row_slice
            ]
        )

        values = np.asarray(
            values,
            dtype=object
        )

        values[
            mask
        ] = None

        return values

    raise ValueError(
        f"Unsupported obs column: "
        f"{name}"
    )


def to_bool_array(
    arr
):

    arr = np.asarray(
        arr
    )

    if arr.dtype == bool:
        return arr

    if np.issubdtype(
        arr.dtype,
        np.number
    ):

        return (
            arr != 0
        )

    return np.fromiter(
        (
            str(x).lower()
            in {
                "true",
                "1",
                "t",
                "yes"
            }
            for x in arr
        ),
        dtype=bool,
        count=len(arr)
    )


def get_n_rows(
    node
):

    if isinstance(
        node,
        h5py.Dataset
    ):

        return node.shape[
            0
        ]

    if "codes" in node:

        return node[
            "codes"
        ].shape[
            0
        ]

    if "values" in node:

        return node[
            "values"
        ].shape[
            0
        ]

    raise ValueError(
        "Cannot determine row count."
    )


# ============================================================
# 10. Sparse counts reader
# ============================================================

def read_counts_rows(
    counts_node,
    start,
    stop,
    n_vars
):

    # Dense storage
    if isinstance(
        counts_node,
        h5py.Dataset
    ):

        arr = np.asarray(
            counts_node[
                start:stop,
                :
            ]
        )

        return sp.csr_matrix(
            arr
        )

    encoding = (
        counts_node.attrs.get(
            "encoding-type",
            ""
        )
    )

    if isinstance(
        encoding,
        bytes
    ):

        encoding = (
            encoding.decode(
                "utf-8"
            )
        )


    # CSR sparse storage
    if (
        "csr" in str(
            encoding
        ).lower()
        or
        (
            "data" in counts_node
            and
            "indices" in counts_node
            and
            "indptr" in counts_node
        )
    ):

        indptr = np.asarray(
            counts_node[
                "indptr"
            ][
                start:
                stop + 1
            ],
            dtype=np.int64
        )

        lo = int(
            indptr[
                0
            ]
        )

        hi = int(
            indptr[
                -1
            ]
        )

        data = np.asarray(
            counts_node[
                "data"
            ][
                lo:hi
            ]
        )

        indices = np.asarray(
            counts_node[
                "indices"
            ][
                lo:hi
            ],
            dtype=np.int32
        )

        local_indptr = (
            indptr
            -
            lo
        )

        return sp.csr_matrix(
            (
                data,
                indices,
                local_indptr
            ),
            shape=(
                stop - start,
                n_vars
            )
        )

    raise ValueError(
        "counts layer is not "
        "dense/CSR storage."
    )


# ============================================================
# 11. Aggregate counts in chunks
# ============================================================

agg_sum = np.zeros(
    (
        n_groups,
        len(genes)
    ),
    dtype=np.float64
)

agg_ncells = np.zeros(
    n_groups,
    dtype=np.int64
)


CHUNK = 50_000


with h5py.File(
    big_path,
    "r"
) as f:

    og = f[
        "obs"
    ]

    counts_node = f[
        "layers"
    ][
        "counts"
    ]

    n_total = get_n_rows(
        og[
            "polygon_flags"
        ]
    )

    print(
        "\nTotal cells:",
        n_total
    )

    print(
        "Processing in chunks of",
        CHUNK
    )

    for start in range(
        0,
        n_total,
        CHUNK
    ):

        stop = min(
            start + CHUNK,
            n_total
        )

        sl = slice(
            start,
            stop
        )

        # --------------------------
        # metadata first
        # --------------------------

        in_poly = to_bool_array(
            read_obs_slice(
                og,
                "is_in_polygon",
                sl
            )
        )

        poly = read_obs_slice(
            og,
            "polygon_flags",
            sl
        )

        ct = read_obs_slice(
            og,
            "celltype_l1",
            sl
        )


        mask = (
            in_poly
            &
            np.isin(
                poly,
                list(
                    eligible_roi_ids
                )
            )
            &
            np.isin(
                ct,
                [
                    "MAC",
                    "FIB"
                ]
            )
        )


        selected_rows = np.flatnonzero(
            mask
        )


        if len(
            selected_rows
        ) > 0:

            # map each selected cell
            # to its ROI×celltype group

            codes = np.fromiter(
                (
                    group_map[
                        (
                            str(
                                poly[j]
                            ),
                            str(
                                ct[j]
                            )
                        )
                    ]
                    for j
                    in selected_rows
                    if (
                        str(
                            poly[j]
                        ),
                        str(
                            ct[j]
                        )
                    )
                    in group_map
                ),
                dtype=np.int64
            )


            # Need exact selected rows
            # corresponding to valid codes

            valid_selected = np.array(
                [
                    j
                    for j
                    in selected_rows
                    if (
                        str(
                            poly[j]
                        ),
                        str(
                            ct[j]
                        )
                    )
                    in group_map
                ],
                dtype=np.int64
            )


            if len(
                valid_selected
            ) > 0:

                X_chunk = (
                    read_counts_rows(
                        counts_node,
                        start,
                        stop,
                        len(genes)
                    )
                )

                X_sel = (
                    X_chunk[
                        valid_selected,
                        :
                    ]
                )


                G = sp.csr_matrix(
                    (
                        np.ones(
                            len(codes),
                            dtype=float
                        ),
                        (
                            codes,
                            np.arange(
                                len(codes)
                            )
                        )
                    ),
                    shape=(
                        n_groups,
                        len(codes)
                    )
                )


                group_sum = (
                    G @ X_sel
                )


                agg_sum += (
                    group_sum
                    .toarray()
                )


                agg_ncells += (
                    np.bincount(
                        codes,
                        minlength=n_groups
                    )
                )


        if (
            stop % 250_000 == 0
            or
            stop == n_total
        ):

            print(
                f"{stop:,} / "
                f"{n_total:,}"
            )


# ============================================================
# 12. Sanity check against metadata counts
# ============================================================

eligible[
    "observed_ncells"
] = (
    agg_ncells
)

eligible[
    "count_difference"
] = (
    eligible[
        "observed_ncells"
    ]
    -
    eligible[
        "n_cells"
    ]
)


print(
    "\nMax absolute cell-count mismatch:"
)

print(
    np.abs(
        eligible[
            "count_difference"
        ]
    ).max()
)


# ============================================================
# 13. Convert SUM to mean counts
# ============================================================

mean_counts = (
    agg_sum
    /
    agg_ncells[
        :,
        None
    ]
)


# ============================================================
# 14. Build output table
# ============================================================

expr_df = eligible[
    [
        "roi_id",
        "celltype_l1",
        "n_cells",
        "Disease",
        "Patient",
        "PC1"
    ]
].copy()


gene_df = pd.DataFrame(
    mean_counts.astype(
        np.float32
    ),
    columns=genes
)


expr_df = pd.concat(
    [
        expr_df.reset_index(
            drop=True
        ),
        gene_df
    ],
    axis=1
)


# ============================================================
# 15. Save
# ============================================================

expr_df.to_csv(
    output_path,
    index=False
)

coverage_audit.to_csv(
    base /
    "figure5_MAC_FIB_cell_coverage_audit.csv",
    index=False
)


print(
    "\n================================"
)

print(
    "FIGURE 5 EXPRESSION EXTRACTION COMPLETE"
)

print(
    "================================"
)

print(
    "\nOutput shape:"
)

print(
    expr_df.shape
)

print(
    "\nUnits by cell type:"
)

print(
    expr_df[
        "celltype_l1"
    ].value_counts()
)

print(
    "\nPatients:"
)

print(
    expr_df.groupby(
        [
            "celltype_l1",
            "Disease"
        ],
        observed=True
    )[
        "Patient"
    ].nunique()
)

print(
    "\nSaved:"
)

print(
    output_path
)

In [ ]:
# ============================================================
# FIGURE 5 — PART 2
# MAC/FIB MOLECULAR TRAJECTORY SCREEN
#
# LN vs anti-GBM
# patient-balanced
# patient-clustered
# spline(PC1) × Disease
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

from patsy import bs
from statsmodels.stats.multitest import multipletests


# ============================================================
# 0. Load extracted expression
# ============================================================

base = Path(str(PROJECT_DIR))

expr_path = (
    base /
    "figure5_MAC_FIB_roi_mean_counts.csv"
)

expr_df = pd.read_csv(
    expr_path
)


metadata_cols = [
    "roi_id",
    "celltype_l1",
    "n_cells",
    "Disease",
    "Patient",
    "PC1"
]

gene_cols = [
    c
    for c in expr_df.columns
    if c not in metadata_cols
]


print(
    "Expression table:",
    expr_df.shape
)

print(
    "Genes:",
    len(gene_cols)
)


# ============================================================
# 1. Normalize each ROI×celltype profile
#
# mean counts
# → total=10,000
# → log1p
# ============================================================

X_raw = (
    expr_df[
        gene_cols
    ]
    .to_numpy(
        dtype=float
    )
)

row_sum = (
    X_raw.sum(
        axis=1
    )
)

if (
    row_sum <= 0
).any():

    raise ValueError(
        "存在总表达量为0的 "
        "ROI×celltype unit。"
    )


X_norm = (
    X_raw
    /
    row_sum[
        :,
        None
    ]
    *
    10000.0
)

X_log = np.log1p(
    X_norm
)


# ============================================================
# 2. Molecular pre-filter
#
# Gene must be detected in:
# >=10% of ROI units overall
# AND >=5 ROIs in each disease
#
# This is expression-based,
# NOT significance-based.
# ============================================================

MIN_DETECTION_FRAC = 0.10

result_rows = []


# ============================================================
# 3. Run MAC and FIB separately
# ============================================================

for celltype in [
    "MAC",
    "FIB"
]:

    idx = (
        expr_df[
            "celltype_l1"
        ]
        .astype(str)
        .to_numpy()
        == celltype
    )


    meta = (
        expr_df.loc[
            idx,
            metadata_cols
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    raw_ct = (
        X_raw[
            idx,
            :
        ]
    )


    log_ct = (
        X_log[
            idx,
            :
        ]
    )


    diseases = (
        meta[
            "Disease"
        ]
        .astype(str)
        .to_numpy()
    )


    print(
        "\n================================"
    )

    print(
        celltype
    )

    print(
        "================================"
    )

    print(
        "ROI units:",
        len(meta)
    )

    print(
        "Patients:",
        meta[
            "Patient"
        ].nunique()
    )


    # patient-balanced weights
    n_roi_patient = (
        meta.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
        .to_numpy()
    )


    weights = (
        1.0 /
        n_roi_patient
    )


    for j, gene in enumerate(
        gene_cols
    ):

        raw_y = (
            raw_ct[
                :,
                j
            ]
        )


        detected = (
            raw_y > 0
        )


        detection_frac = (
            detected.mean()
        )


        detected_LN = int(
            detected[
                diseases
                == "SLE"
            ].sum()
        )


        detected_GBM = int(
            detected[
                diseases
                == "GBM"
            ].sum()
        )


        if (
            detection_frac
            < MIN_DETECTION_FRAC
            or
            detected_LN < 5
            or
            detected_GBM < 5
        ):

            continue


        y = (
            log_ct[
                :,
                j
            ]
        )


        # skip constant genes
        if (
            np.nanstd(
                y
            )
            < 1e-10
        ):

            continue


        d = meta[
            [
                "Disease",
                "Patient",
                "PC1"
            ]
        ].copy()


        d[
            "y"
        ] = y


        d[
            "patient_weight"
        ] = weights


        d[
            "Disease"
        ] = pd.Categorical(
            d[
                "Disease"
            ].astype(str),
            categories=[
                "SLE",
                "GBM"
            ]
        )


        try:

            fit = smf.wls(
                (
                    "y ~ "
                    "bs(PC1, df=3, degree=3, "
                    "include_intercept=False) "
                    "* C(Disease)"
                ),
                data=d,
                weights=d[
                    "patient_weight"
                ]
            ).fit(
                cov_type="cluster",
                cov_kwds={
                    "groups":
                        d[
                            "Patient"
                        ]
                }
            )


            terms = [
                term
                for term
                in fit.params.index
                if ":" in term
                and
                "C(Disease)"
                in term
                and
                "bs(PC1"
                in term
            ]


            R = np.zeros(
                (
                    len(terms),
                    len(
                        fit.params
                    )
                )
            )


            for k, term in enumerate(
                terms
            ):

                R[
                    k,
                    fit.params.index
                    .get_loc(
                        term
                    )
                ] = 1


            wt = fit.wald_test(
                R,
                scalar=True
            )


            pvalue = float(
                wt.pvalue
            )


            rank = int(
                np.linalg.matrix_rank(
                    fit.model.exog
                )
            )


            ncols = int(
                fit.model.exog.shape[
                    1
                ]
            )


            # --------------------------------
            # Quantify trajectory divergence
            # --------------------------------

            grid = np.linspace(
                d[
                    "PC1"
                ].min(),
                d[
                    "PC1"
                ].max(),
                100
            )


            predictions = {}


            for disease in [
                "SLE",
                "GBM"
            ]:

                newdata = pd.DataFrame({
                    "PC1":
                        grid,

                    "Disease":
                        pd.Categorical(
                            [disease]
                            *
                            len(grid),

                            categories=[
                                "SLE",
                                "GBM"
                            ]
                        )
                })


                pred = (
                    fit.predict(
                        newdata
                    )
                )


                predictions[
                    disease
                ] = np.asarray(
                    pred
                )


            diff = (
                predictions[
                    "GBM"
                ]
                -
                predictions[
                    "SLE"
                ]
            )


            max_abs_curve_diff = float(
                np.max(
                    np.abs(
                        diff
                    )
                )
            )


            late_mask = (
                grid
                >=
                np.quantile(
                    grid,
                    0.75
                )
            )


            late_GBM_minus_LN = float(
                np.mean(
                    diff[
                        late_mask
                    ]
                )
            )


            result_rows.append({
                "celltype":
                    celltype,

                "gene":
                    gene,

                "pvalue":
                    pvalue,

                "matrix_rank":
                    rank,

                "n_columns":
                    ncols,

                "n_ROI":
                    len(d),

                "n_patients":
                    d[
                        "Patient"
                    ].nunique(),

                "n_GBM_patients":
                    d.loc[
                        d[
                            "Disease"
                        ]
                        .astype(str)
                        == "GBM",
                        "Patient"
                    ].nunique(),

                "detection_fraction":
                    detection_frac,

                "detected_LN_ROI":
                    detected_LN,

                "detected_GBM_ROI":
                    detected_GBM,

                "max_abs_curve_diff":
                    max_abs_curve_diff,

                "late_GBM_minus_LN":
                    late_GBM_minus_LN
            })


        except Exception as e:

            print(
                "Skipped:",
                celltype,
                gene,
                str(e)[
                    :100
                ]
            )


# ============================================================
# 4. Results table
# ============================================================

gene_results = pd.DataFrame(
    result_rows
)


if len(
    gene_results
) == 0:

    raise ValueError(
        "没有基因通过表达过滤。"
    )


# ============================================================
# 5. BH-FDR within each cell type
# ============================================================

gene_results[
    "FDR_within_celltype"
] = (
    gene_results
    .groupby(
        "celltype",
        observed=True
    )[
        "pvalue"
    ]
    .transform(
        lambda x:
        multipletests(
            x,
            method="fdr_bh"
        )[1]
    )
)


# ============================================================
# 6. Global BH across all MAC+FIB tests
# ============================================================

gene_results[
    "FDR_global"
] = multipletests(
    gene_results[
        "pvalue"
    ],
    method="fdr_bh"
)[1]


gene_results = (
    gene_results
    .sort_values(
        [
            "FDR_global",
            "FDR_within_celltype",
            "pvalue"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 7. Save
# ============================================================

result_path = (
    base /
    "figure5_MAC_FIB_gene_trajectory_results.csv"
)

gene_results.to_csv(
    result_path,
    index=False
)


# ============================================================
# 8. Summary
# ============================================================

print(
    "\n================================"
)

print(
    "FIGURE 5 MOLECULAR SCREEN COMPLETE"
)

print(
    "================================"
)


print(
    "\nGenes tested:"
)

print(
    gene_results[
        "celltype"
    ].value_counts()
)


print(
    "\nGenes with GLOBAL FDR < 0.05:"
)

print(
    gene_results[
        gene_results[
            "FDR_global"
        ] < 0.05
    ]
    .groupby(
        "celltype",
        observed=True
    )
    .size()
)


print(
    "\nNon-full-rank models:"
)

display(
    gene_results[
        gene_results[
            "matrix_rank"
        ]
        !=
        gene_results[
            "n_columns"
        ]
    ]
)


print(
    "\n=============================="
)

print(
    "TOP 20 MAC"
)

print(
    "=============================="
)

display(
    gene_results[
        gene_results[
            "celltype"
        ]
        == "MAC"
    ].head(
        20
    )
)


print(
    "\n=============================="
)

print(
    "TOP 20 FIB"
)

print(
    "=============================="
)

display(
    gene_results[
        gene_results[
            "celltype"
        ]
        == "FIB"
    ].head(
        20
    )
)


print(
    "\nSaved:"
)

print(
    result_path
)

In [ ]:
# ============================================================
# FIGURE 5 — CRITICAL TECHNICAL AUDIT
#
# Check:
# 1. big h5ad gene order vs ROI h5ad gene order
# 2. counts sparse encoding: CSR or CSC
# 3. basic marker sanity
# ============================================================

from pathlib import Path

import h5py
import anndata as ad
import pandas as pd
import numpy as np


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

big_path = (
    base /
    "GSE294965_processed_data.h5ad"
)

roi_path = (
    base /
    "roi_782_PC1_primary.h5ad"
)

expr_path = (
    base /
    "figure5_MAC_FIB_roi_mean_counts.csv"
)

result_path = (
    base /
    "figure5_MAC_FIB_gene_trajectory_results.csv"
)


# ============================================================
# 1. Read small ROI object
# ============================================================

roi_ad = ad.read_h5ad(
    roi_path
)

roi_genes = (
    roi_ad.var_names
    .astype(str)
    .to_numpy()
)

print(
    "ROI genes:",
    len(roi_genes)
)


# ============================================================
# 2. Read ONLY big-file var metadata
#    var has only 480 rows, so this is safe
# ============================================================

try:
    from anndata.io import read_elem
except ImportError:
    from anndata._io.specs import read_elem


with h5py.File(
    big_path,
    "r"
) as f:

    big_var = read_elem(
        f["var"]
    )

    big_genes = (
        big_var.index
        .astype(str)
        .to_numpy()
    )

    counts_node = (
        f["layers"]["counts"]
    )

    encoding = (
        counts_node.attrs.get(
            "encoding-type",
            "UNKNOWN"
        )
    )

    if isinstance(
        encoding,
        bytes
    ):
        encoding = encoding.decode(
            "utf-8"
        )

    stored_shape = (
        counts_node.attrs.get(
            "shape",
            "UNKNOWN"
        )
    )


print(
    "\n================================"
)

print(
    "GENE ORDER AUDIT"
)

print(
    "================================"
)

print(
    "Big h5ad genes:",
    len(big_genes)
)

print(
    "ROI h5ad genes:",
    len(roi_genes)
)

print(
    "Same gene SET:",
    set(big_genes)
    ==
    set(roi_genes)
)

print(
    "Exact same ORDER:",
    np.array_equal(
        big_genes,
        roi_genes
    )
)


# ============================================================
# 3. Show first mismatches if order differs
# ============================================================

if (
    len(big_genes)
    ==
    len(roi_genes)
):

    mismatch_idx = np.where(
        big_genes
        !=
        roi_genes
    )[0]

    print(
        "\nNumber of order mismatches:",
        len(mismatch_idx)
    )

    if len(
        mismatch_idx
    ) > 0:

        print(
            "\nFirst 20 mismatches:"
        )

        for i in mismatch_idx[
            :20
        ]:

            print(
                i,
                "BIG =",
                big_genes[i],
                "| ROI =",
                roi_genes[i]
            )


# ============================================================
# 4. Counts encoding
# ============================================================

print(
    "\n================================"
)

print(
    "COUNTS STORAGE AUDIT"
)

print(
    "================================"
)

print(
    "layers/counts encoding-type:",
    encoding
)

print(
    "layers/counts stored shape:",
    stored_shape
)

print(
    "Node type:",
    type(counts_node)
)


# ============================================================
# 5. Load extracted Figure 5 expression
# ============================================================

expr_df = pd.read_csv(
    expr_path
)

metadata_cols = [
    "roi_id",
    "celltype_l1",
    "n_cells",
    "Disease",
    "Patient",
    "PC1"
]

gene_cols = [
    c for c in expr_df.columns
    if c not in metadata_cols
]


print(
    "\nExtracted gene columns:",
    len(gene_cols)
)

print(
    "Extracted genes exactly equal ROI genes:",
    np.array_equal(
        np.array(
            gene_cols,
            dtype=str
        ),
        roi_genes
    )
)


# ============================================================
# 6. Marker sanity panel
# ============================================================

marker_candidates = [
    # macrophage / myeloid
    "C1QA",
    "C1QB",
    "C1QC",
    "CD68",
    "LST1",
    "TYROBP",
    "FCGR3A",
    "ITGAM",
    "SIRPA",
    "S100A9",

    # fibroblast / ECM
    "COL1A1",
    "COL1A2",
    "COL3A1",
    "DCN",
    "LUM",
    "COL6A1",
    "COL6A2",
    "CXCL12",
    "COL16A1",

    # possible contamination indicators
    "FGFBP2",
    "GZMA",
    "THEMIS",
    "CD3D",
    "KLRK1",
    "TPSAB1",
    "PLVAP",
    "AQP2",
    "CUBN"
]


available_markers = [
    g for g
    in marker_candidates
    if g in gene_cols
]


print(
    "\n================================"
)

print(
    "AVAILABLE AUDIT MARKERS"
)

print(
    "================================"
)

print(
    available_markers
)


# mean raw-count signal per ROI×celltype unit
marker_summary = (
    expr_df
    .groupby(
        "celltype_l1",
        observed=True
    )[
        available_markers
    ]
    .mean()
    .T
)


print(
    "\nMean expression by annotated cell type:"
)

display(
    marker_summary
)


# ============================================================
# 7. Significant-gene counts
# ============================================================

gene_results = pd.read_csv(
    result_path
)

print(
    "\n================================"
)

print(
    "SIGNIFICANT GENE COUNTS"
)

print(
    "================================"
)


sig_counts = (
    gene_results.assign(
        global_sig=
        gene_results[
            "FDR_global"
        ] < 0.05
    )
    .groupby(
        "celltype",
        observed=True
    )
    .agg(
        genes_tested=(
            "gene",
            "size"
        ),
        global_FDR_lt_005=(
            "global_sig",
            "sum"
        )
    )
)

display(
    sig_counts
)

In [ ]:
# ============================================================
# FIGURE 5 — CELL-TYPE SPECIFICITY AUDIT
#
# Goal:
# Separate statistically significant genes from
# biologically cell-type-consistent candidates
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

expr_df = pd.read_csv(
    base /
    "figure5_MAC_FIB_roi_mean_counts.csv"
)

gene_results = pd.read_csv(
    base /
    "figure5_MAC_FIB_gene_trajectory_results.csv"
)


# ============================================================
# 1. Define metadata/gene columns
# ============================================================

metadata_cols = [
    "roi_id",
    "celltype_l1",
    "n_cells",
    "Disease",
    "Patient",
    "PC1"
]

gene_cols = [
    c
    for c in expr_df.columns
    if c not in metadata_cols
]


# ============================================================
# 2. Mean expression in MAC and FIB
# ============================================================

mean_expr = (
    expr_df
    .groupby(
        "celltype_l1",
        observed=True
    )[gene_cols]
    .mean()
    .T
)

mean_expr = mean_expr.rename(
    columns={
        "MAC": "mean_MAC",
        "FIB": "mean_FIB"
    }
)


# ============================================================
# 3. Specificity score
#
# Positive = more highly expressed in MAC
# Negative = more highly expressed in FIB
#
# Only for interpretation.
# Does NOT change statistical significance.
# ============================================================

eps = 1e-4

mean_expr[
    "MAC_vs_FIB_log2ratio"
] = np.log2(
    (
        mean_expr["mean_MAC"]
        + eps
    )
    /
    (
        mean_expr["mean_FIB"]
        + eps
    )
)


mean_expr = (
    mean_expr
    .reset_index()
    .rename(
        columns={
            "index":
                "gene"
        }
    )
)


# ============================================================
# 4. Merge with trajectory statistics
# ============================================================

audit = gene_results.merge(
    mean_expr,
    on="gene",
    how="left"
)


# ============================================================
# 5. Cell-type consistency
#
# Very conservative rule:
#
# MAC candidate:
# mean_MAC > mean_FIB
#
# FIB candidate:
# mean_FIB > mean_MAC
#
# No arbitrary fold-change cutoff.
# ============================================================

audit[
    "celltype_consistent"
] = np.where(
    audit["celltype"] == "MAC",

    audit["mean_MAC"]
    >
    audit["mean_FIB"],

    audit["mean_FIB"]
    >
    audit["mean_MAC"]
)


# ============================================================
# 6. Significant + cell-type-consistent candidates
# ============================================================

candidates = audit[
    (
        audit["FDR_global"]
        < 0.05
    )
    &
    (
        audit[
            "celltype_consistent"
        ]
    )
].copy()


# ============================================================
# 7. Ranking score
#
# We do NOT invent another significance threshold.
#
# rank:
# 1) global FDR
# 2) trajectory effect size
# ============================================================

candidates = (
    candidates
    .sort_values(
        [
            "celltype",
            "FDR_global",
            "max_abs_curve_diff"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)


# ============================================================
# 8. Summary
# ============================================================

print(
    "================================"
)

print(
    "CELL-TYPE CONSISTENCY SUMMARY"
)

print(
    "================================"
)


summary = (
    audit[
        audit["FDR_global"]
        < 0.05
    ]
    .groupby(
        "celltype",
        observed=True
    )
    .agg(
        significant_genes=(
            "gene",
            "size"
        ),

        celltype_consistent=(
            "celltype_consistent",
            "sum"
        )
    )
)


summary[
    "fraction_consistent"
] = (
    summary[
        "celltype_consistent"
    ]
    /
    summary[
        "significant_genes"
    ]
)


display(
    summary
)


# ============================================================
# 9. TOP 20 MAC candidates
# ============================================================

columns_to_show = [
    "gene",
    "FDR_global",
    "detection_fraction",
    "max_abs_curve_diff",
    "late_GBM_minus_LN",
    "mean_MAC",
    "mean_FIB",
    "MAC_vs_FIB_log2ratio"
]


print(
    "\n=============================="
)

print(
    "TOP MAC CELL-TYPE-CONSISTENT CANDIDATES"
)

print(
    "=============================="
)


mac_candidates = (
    candidates[
        candidates[
            "celltype"
        ]
        == "MAC"
    ]
)


display(
    mac_candidates[
        columns_to_show
    ].head(
        20
    )
)


# ============================================================
# 10. TOP 20 FIB candidates
# ============================================================

print(
    "\n=============================="
)

print(
    "TOP FIB CELL-TYPE-CONSISTENT CANDIDATES"
)

print(
    "=============================="
)


fib_candidates = (
    candidates[
        candidates[
            "celltype"
        ]
        == "FIB"
    ]
)


display(
    fib_candidates[
        columns_to_show
    ].head(
        20
    )
)


# ============================================================
# 11. Save
# ============================================================

audit.to_csv(
    base /
    "figure5_gene_celltype_specificity_audit.csv",
    index=False
)


candidates.to_csv(
    base /
    "figure5_celltype_consistent_gene_candidates.csv",
    index=False
)


print(
    "\nSaved."
)

In [ ]:
# ============================================================
# FIGURE 5 — MOLECULAR TRAJECTORY PROGRAM CLUSTERING
#
# Significant + cell-type-consistent genes
# clustered by anti-GBM minus LN trajectory shape
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from patsy import bs
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.spatial.distance import pdist


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

figdir = base / "figures"

figdir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. Load data
# ============================================================

expr_df = pd.read_csv(
    base /
    "figure5_MAC_FIB_roi_mean_counts.csv"
)

candidates = pd.read_csv(
    base /
    "figure5_celltype_consistent_gene_candidates.csv"
)


metadata_cols = [
    "roi_id",
    "celltype_l1",
    "n_cells",
    "Disease",
    "Patient",
    "PC1"
]


gene_cols = [
    c for c in expr_df.columns
    if c not in metadata_cols
]


print(
    "Expression data:",
    expr_df.shape
)

print(
    "\nCandidate genes:"
)

print(
    candidates[
        "celltype"
    ].value_counts()
)


# ============================================================
# 2. Reproduce normalized log expression
# ============================================================

X_raw = (
    expr_df[
        gene_cols
    ]
    .to_numpy(
        dtype=float
    )
)


row_sum = (
    X_raw.sum(
        axis=1
    )
)


X_norm = (
    X_raw
    /
    row_sum[
        :,
        None
    ]
    *
    10000.0
)


X_log = np.log1p(
    X_norm
)


gene_index = {
    gene: i
    for i, gene
    in enumerate(
        gene_cols
    )
}


# ============================================================
# 3. Function:
# formal patient-balanced spline trajectory
# ============================================================

def fit_gene_trajectory(
    celltype,
    gene,
    n_grid=100
):

    if gene not in gene_index:
        return None


    ct_mask = (
        expr_df[
            "celltype_l1"
        ]
        .astype(str)
        .to_numpy()
        ==
        celltype
    )


    meta = (
        expr_df.loc[
            ct_mask,
            metadata_cols
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    gene_position = (
        gene_index[
            gene
        ]
    )


    y = (
        X_log[
            ct_mask,
            gene_position
        ]
    )


    d = meta[
        [
            "Disease",
            "Patient",
            "PC1"
        ]
    ].copy()


    d[
        "y"
    ] = y


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ]
        .astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    # --------------------------
    # Patient-balanced weights
    # --------------------------

    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )


    # --------------------------
    # Formal model
    # --------------------------

    fit = smf.wls(
        (
            "y ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    # --------------------------
    # Prediction grid
    # --------------------------

    grid = np.linspace(
        d[
            "PC1"
        ].min(),
        d[
            "PC1"
        ].max(),
        n_grid
    )


    predictions = {}


    for disease in [
        "SLE",
        "GBM"
    ]:

        newdata = pd.DataFrame({
            "PC1":
                grid,

            "Disease":
                pd.Categorical(
                    [disease]
                    *
                    len(grid),

                    categories=[
                        "SLE",
                        "GBM"
                    ]
                )
        })


        predictions[
            disease
        ] = np.asarray(
            fit.predict(
                newdata
            )
        )


    # anti-GBM minus LN
    delta = (
        predictions[
            "GBM"
        ]
        -
        predictions[
            "SLE"
        ]
    )


    return {
        "grid":
            grid,

        "LN":
            predictions[
                "SLE"
            ],

        "GBM":
            predictions[
                "GBM"
            ],

        "delta":
            delta
    }


# ============================================================
# 4. Build trajectory matrices
# ============================================================

trajectory_results = {}

module_tables = []


for celltype in [
    "MAC",
    "FIB"
]:

    genes = (
        candidates.loc[
            candidates[
                "celltype"
            ]
            == celltype,
            "gene"
        ]
        .astype(str)
        .tolist()
    )


    curve_list = []

    kept_genes = []

    raw_delta_list = []


    print(
        "\nProcessing",
        celltype,
        ":",
        len(genes),
        "genes"
    )


    for i, gene in enumerate(
        genes,
        start=1
    ):

        result = (
            fit_gene_trajectory(
                celltype,
                gene,
                n_grid=100
            )
        )


        if result is None:
            continue


        delta = np.asarray(
            result[
                "delta"
            ],
            dtype=float
        )


        # Skip pathological curves
        if (
            not np.all(
                np.isfinite(
                    delta
                )
            )
            or
            np.std(
                delta
            )
            < 1e-8
        ):

            continue


        # --------------------------------
        # z-score within gene:
        # cluster TRAJECTORY SHAPE,
        # not absolute expression magnitude
        # --------------------------------

        z = (
            delta
            -
            np.mean(
                delta
            )
        ) / np.std(
            delta
        )


        curve_list.append(
            z
        )


        raw_delta_list.append(
            delta
        )


        kept_genes.append(
            gene
        )


    Zmat = np.vstack(
        curve_list
    )


    raw_mat = np.vstack(
        raw_delta_list
    )


    # ========================================================
    # 5. Hierarchical clustering
    # ========================================================

    distances = pdist(
        Zmat,
        metric="correlation"
    )


    linkage_matrix = linkage(
        distances,
        method="average"
    )


    order = leaves_list(
        linkage_matrix
    )


    # descriptive 3-module cut
    modules = fcluster(
        linkage_matrix,
        t=3,
        criterion="maxclust"
    )


    # ========================================================
    # 6. Save module assignments
    # ========================================================

    temp = pd.DataFrame({
        "celltype":
            celltype,

        "gene":
            kept_genes,

        "module":
            modules,

        "max_abs_delta":
            np.max(
                np.abs(
                    raw_mat
                ),
                axis=1
            ),

        "mean_delta":
            np.mean(
                raw_mat,
                axis=1
            ),

        "late_delta":
            np.mean(
                raw_mat[
                    :,
                    75:
                ],
                axis=1
            )
    })


    # attach formal FDR
    temp = temp.merge(
        candidates[
            [
                "celltype",
                "gene",
                "FDR_global",
                "detection_fraction",
                "mean_MAC",
                "mean_FIB"
            ]
        ],
        on=[
            "celltype",
            "gene"
        ],
        how="left"
    )


    module_tables.append(
        temp
    )


    trajectory_results[
        celltype
    ] = {
        "genes":
            np.array(
                kept_genes
            ),

        "zmat":
            Zmat,

        "raw":
            raw_mat,

        "order":
            order,

        "modules":
            modules,

        "grid":
            result[
                "grid"
            ]
    }


# ============================================================
# 7. Combined module table
# ============================================================

module_df = pd.concat(
    module_tables,
    ignore_index=True
)


print(
    "\n================================"
)

print(
    "MODULE SIZES"
)

print(
    "================================"
)


module_sizes = (
    module_df
    .groupby(
        [
            "celltype",
            "module"
        ],
        observed=True
    )
    .size()
    .rename(
        "n_genes"
    )
)


display(
    module_sizes
)


# ============================================================
# 8. Representative genes
#
# Within each module:
# choose genes with largest trajectory amplitude.
#
# For descriptive visualization only.
# ============================================================

representatives = (
    module_df
    .sort_values(
        "max_abs_delta",
        ascending=False
    )
    .groupby(
        [
            "celltype",
            "module"
        ],
        observed=True
    )
    .head(
        8
    )
    .sort_values(
        [
            "celltype",
            "module",
            "max_abs_delta"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
)


print(
    "\n================================"
)

print(
    "TOP REPRESENTATIVE GENES"
)

print(
    "================================"
)


display(
    representatives[
        [
            "celltype",
            "module",
            "gene",
            "FDR_global",
            "max_abs_delta",
            "late_delta",
            "detection_fraction"
        ]
    ]
)


# ============================================================
# 9. Heatmaps
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(
        14,
        10
    )
)


for ax, celltype in zip(
    axes,
    [
        "MAC",
        "FIB"
    ]
):

    res = trajectory_results[
        celltype
    ]


    ordered = (
        res[
            "zmat"
        ][
            res[
                "order"
            ],
            :
        ]
    )


    ordered_genes = (
        res[
            "genes"
        ][
            res[
                "order"
            ]
        ]
    )


    im = ax.imshow(
        ordered,
        aspect="auto",
        interpolation="nearest",
        cmap="coolwarm",
        vmin=-2.5,
        vmax=2.5,
        origin="upper"
    )


    grid = res[
        "grid"
    ]


    # x-axis positions
    tick_positions = np.linspace(
        0,
        len(grid) - 1,
        5
    )


    tick_labels = np.linspace(
        grid.min(),
        grid.max(),
        5
    )


    ax.set_xticks(
        tick_positions
    )


    ax.set_xticklabels(
        [
            f"{x:.1f}"
            for x in tick_labels
        ]
    )


    ax.set_xlabel(
        "Crescent progression (PC1)"
    )


    ax.set_ylabel(
        "Trajectory-divergent genes"
    )


    ax.set_title(
        (
            f"{celltype}: anti-GBM − LN\n"
            "row-standardized trajectory difference"
        ),
        fontsize=12
    )


    # only label representative genes
    rep_genes = set(
        representatives.loc[
            representatives[
                "celltype"
            ]
            == celltype,
            "gene"
        ]
    )


    yticks = []

    ylabels = []


    for position, gene in enumerate(
        ordered_genes
    ):

        if gene in rep_genes:

            yticks.append(
                position
            )

            ylabels.append(
                gene
            )


    ax.set_yticks(
        yticks
    )


    ax.set_yticklabels(
        ylabels,
        fontsize=7
    )


# common colorbar
cbar = fig.colorbar(
    im,
    ax=axes.ravel().tolist(),
    fraction=0.025,
    pad=0.02
)


cbar.set_label(
    "Within-gene standardized\nanti-GBM − LN difference"
)


plt.subplots_adjust(
    wspace=0.28,
    right=0.90
)


heatmap_path = (
    figdir /
    "Figure5_SCREEN_MAC_FIB_molecular_trajectory_heatmaps.png"
)


plt.savefig(
    heatmap_path,
    dpi=500,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


# ============================================================
# 10. Save module assignments
# ============================================================

module_df.to_csv(
    base /
    "figure5_MAC_FIB_trajectory_modules.csv",
    index=False
)


representatives.to_csv(
    base /
    "figure5_MAC_FIB_module_representative_genes.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    heatmap_path
)

In [ ]:
# ============================================================
# FIGURE 5 — MODULE TRAJECTORY SUMMARY
#
# For each MAC/FIB molecular module:
# - refit formal trajectories
# - calculate anti-GBM minus LN curve
# - show individual genes
# - show module median trajectory
# - quantify within-module coherence
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from patsy import bs


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

figdir = base / "figures"

figdir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. Load data
# ============================================================

expr_df = pd.read_csv(
    base /
    "figure5_MAC_FIB_roi_mean_counts.csv"
)

module_df = pd.read_csv(
    base /
    "figure5_MAC_FIB_trajectory_modules.csv"
)


metadata_cols = [
    "roi_id",
    "celltype_l1",
    "n_cells",
    "Disease",
    "Patient",
    "PC1"
]


gene_cols = [
    c
    for c in expr_df.columns
    if c not in metadata_cols
]


print(
    "Expression table:",
    expr_df.shape
)

print(
    "\nModule assignments:"
)

print(
    module_df
    .groupby(
        [
            "celltype",
            "module"
        ],
        observed=True
    )
    .size()
)


# ============================================================
# 2. Recreate normalized log expression
# ============================================================

X_raw = (
    expr_df[
        gene_cols
    ]
    .to_numpy(
        dtype=float
    )
)


row_sum = (
    X_raw.sum(
        axis=1
    )
)


X_norm = (
    X_raw
    /
    row_sum[
        :,
        None
    ]
    *
    10000.0
)


X_log = np.log1p(
    X_norm
)


gene_index = {
    gene: i
    for i, gene
    in enumerate(
        gene_cols
    )
}


# ============================================================
# 3. Formal gene trajectory prediction
# ============================================================

def get_delta_curve(
    celltype,
    gene,
    n_grid=120
):

    if gene not in gene_index:
        return None


    mask = (
        expr_df[
            "celltype_l1"
        ]
        .astype(str)
        .to_numpy()
        ==
        celltype
    )


    meta = (
        expr_df.loc[
            mask,
            metadata_cols
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    y = (
        X_log[
            mask,
            gene_index[
                gene
            ]
        ]
    )


    d = meta[
        [
            "Disease",
            "Patient",
            "PC1"
        ]
    ].copy()


    d["y"] = y


    d["Disease"] = pd.Categorical(
        d["Disease"].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    # patient-balanced weights
    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )


    fit = smf.wls(
        (
            "y ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    grid = np.linspace(
        d["PC1"].min(),
        d["PC1"].max(),
        n_grid
    )


    preds = {}


    for disease in [
        "SLE",
        "GBM"
    ]:

        newdata = pd.DataFrame({
            "PC1":
                grid,

            "Disease":
                pd.Categorical(
                    [disease]
                    *
                    len(grid),

                    categories=[
                        "SLE",
                        "GBM"
                    ]
                )
        })


        preds[
            disease
        ] = np.asarray(
            fit.predict(
                newdata
            )
        )


    delta = (
        preds["GBM"]
        -
        preds["SLE"]
    )


    return (
        grid,
        delta
    )


# ============================================================
# 4. Calculate module trajectories
# ============================================================

module_summary_rows = []

representative_rows = []

module_curves = {}


for celltype in [
    "MAC",
    "FIB"
]:

    modules = sorted(
        module_df.loc[
            module_df[
                "celltype"
            ]
            == celltype,
            "module"
        ].unique()
    )


    for module in modules:

        genes = (
            module_df.loc[
                (
                    module_df[
                        "celltype"
                    ]
                    == celltype
                )
                &
                (
                    module_df[
                        "module"
                    ]
                    == module
                ),
                "gene"
            ]
            .astype(str)
            .tolist()
        )


        curves = []

        kept_genes = []

        grid_saved = None


        for gene in genes:

            result = (
                get_delta_curve(
                    celltype,
                    gene,
                    n_grid=120
                )
            )


            if result is None:
                continue


            grid, delta = result


            if (
                not np.all(
                    np.isfinite(
                        delta
                    )
                )
            ):
                continue


            curves.append(
                delta
            )

            kept_genes.append(
                gene
            )

            grid_saved = grid


        raw_mat = np.vstack(
            curves
        )


        # ====================================================
        # Robust module consensus:
        # median anti-GBM - LN trajectory
        # ====================================================

        consensus = np.median(
            raw_mat,
            axis=0
        )


        # ====================================================
        # Within-module coherence
        #
        # Pearson correlation between each gene trajectory
        # and module consensus.
        # ====================================================

        correlations = []


        for curve in raw_mat:

            if (
                np.std(
                    curve
                ) < 1e-10
                or
                np.std(
                    consensus
                ) < 1e-10
            ):

                correlations.append(
                    np.nan
                )

            else:

                correlations.append(
                    np.corrcoef(
                        curve,
                        consensus
                    )[
                        0,
                        1
                    ]
                )


        correlations = np.asarray(
            correlations
        )


        # ====================================================
        # Early / middle / late trajectory values
        # ====================================================

        n = len(
            consensus
        )


        early = float(
            np.mean(
                consensus[
                    : int(
                        n * 0.25
                    )
                ]
            )
        )


        middle = float(
            np.mean(
                consensus[
                    int(
                        n * 0.375
                    ):
                    int(
                        n * 0.625
                    )
                ]
            )
        )


        late = float(
            np.mean(
                consensus[
                    int(
                        n * 0.75
                    ):
                ]
            )
        )


        amplitude = float(
            np.max(
                consensus
            )
            -
            np.min(
                consensus
            )
        )


        median_corr = float(
            np.nanmedian(
                correlations
            )
        )


        frac_corr_05 = float(
            np.nanmean(
                correlations
                >= 0.5
            )
        )


        module_summary_rows.append({
            "celltype":
                celltype,

            "module":
                module,

            "n_genes":
                len(
                    kept_genes
                ),

            "early_delta":
                early,

            "middle_delta":
                middle,

            "late_delta":
                late,

            "trajectory_amplitude":
                amplitude,

            "median_gene_consensus_r":
                median_corr,

            "fraction_genes_r_ge_0.5":
                frac_corr_05
        })


        # ====================================================
        # Most representative genes:
        # highest correlation with module consensus
        # ====================================================

        order = np.argsort(
            np.nan_to_num(
                correlations,
                nan=-999
            )
        )[
            ::-1
        ]


        for rank, idx in enumerate(
            order[
                :8
            ],
            start=1
        ):

            representative_rows.append({
                "celltype":
                    celltype,

                "module":
                    module,

                "rank":
                    rank,

                "gene":
                    kept_genes[
                        idx
                    ],

                "consensus_correlation":
                    correlations[
                        idx
                    ]
            })


        module_curves[
            (
                celltype,
                module
            )
        ] = {
            "grid":
                grid_saved,

            "raw":
                raw_mat,

            "consensus":
                consensus,

            "genes":
                kept_genes,

            "correlations":
                correlations
        }


# ============================================================
# 5. Summary tables
# ============================================================

module_summary = pd.DataFrame(
    module_summary_rows
)


representatives2 = pd.DataFrame(
    representative_rows
)


print(
    "\n================================"
)

print(
    "MODULE TRAJECTORY SUMMARY"
)

print(
    "================================"
)


display(
    module_summary
    .sort_values(
        [
            "celltype",
            "module"
        ]
    )
)


print(
    "\n================================"
)

print(
    "MOST COHERENT REPRESENTATIVE GENES"
)

print(
    "================================"
)


display(
    representatives2
    .sort_values(
        [
            "celltype",
            "module",
            "rank"
        ]
    )
)


# ============================================================
# 6. Plot six module trajectories
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(
        15,
        8
    ),
    sharex=True
)


for row, celltype in enumerate(
    [
        "MAC",
        "FIB"
    ]
):

    modules = sorted(
        module_df.loc[
            module_df[
                "celltype"
            ]
            == celltype,
            "module"
        ].unique()
    )


    for col, module in enumerate(
        modules
    ):

        ax = axes[
            row,
            col
        ]


        res = module_curves[
            (
                celltype,
                module
            )
        ]


        grid = res[
            "grid"
        ]


        raw_mat = res[
            "raw"
        ]


        consensus = res[
            "consensus"
        ]


        # individual genes
        for curve in raw_mat:

            ax.plot(
                grid,
                curve,
                linewidth=0.8,
                alpha=0.15
            )


        # module consensus
        ax.plot(
            grid,
            consensus,
            linewidth=3,
            color="black"
        )


        ax.axhline(
            0,
            linestyle="--",
            linewidth=1
        )


        info = module_summary[
            (
                module_summary[
                    "celltype"
                ]
                == celltype
            )
            &
            (
                module_summary[
                    "module"
                ]
                == module
            )
        ].iloc[0]


        ax.set_title(
            (
                f"{celltype} module {module}"
                "\n"
                f"n={int(info['n_genes'])}, "
                f"median r={info['median_gene_consensus_r']:.2f}"
            )
        )


        ax.set_xlabel(
            "Crescent progression (PC1)"
        )


        ax.set_ylabel(
            "anti-GBM − LN\nlog-expression"
        )


plt.tight_layout()


module_fig = (
    figdir /
    "Figure5_SCREEN_module_consensus_trajectories.png"
)


plt.savefig(
    module_fig,
    dpi=500,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


# ============================================================
# 7. Save
# ============================================================

module_summary.to_csv(
    base /
    "figure5_module_trajectory_summary.csv",
    index=False
)


representatives2.to_csv(
    base /
    "figure5_module_coherent_representatives.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    module_fig
)

In [ ]:
%pip install -q gseapy

In [ ]:
try:
    import gseapy as gp
    print("gseapy installed successfully!")
    print("version:", gp.__version__)
except Exception as e:
    print("gseapy not available")
    print(e)

In [ ]:
# ============================================================
# FIGURE 5 — PANEL-AWARE PATHWAY ENRICHMENT
#
# Universe:
# genes actually tested in this 480-gene Xenium panel
#
# Query:
# trajectory modules defined above
#
# Statistical test:
# hypergeometric ORA + BH correction
#
# ============================================================

from pathlib import Path

import re
import numpy as np
import pandas as pd
import gseapy as gp

from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))


module_df = pd.read_csv(
    base /
    "figure5_MAC_FIB_trajectory_modules.csv"
)


gene_results = pd.read_csv(
    base /
    "figure5_MAC_FIB_gene_trajectory_results.csv"
)


# ============================================================
# 1. Find currently available pathway libraries
#
# We choose the newest available GO BP / Reactome / Hallmark
# automatically instead of hard-coding a year.
# ============================================================

available_libs = gp.get_library_name(
    organism="Human"
)


def latest_library(
    libraries,
    keyword
):

    candidates = [
        x for x in libraries
        if keyword.lower()
        in x.lower()
    ]

    if len(candidates) == 0:
        return None


    def get_year(x):

        years = re.findall(
            r"20\d{2}",
            x
        )

        if len(years) == 0:
            return 0

        return max(
            int(y)
            for y in years
        )


    candidates = sorted(
        candidates,
        key=lambda x:
        (
            get_year(x),
            x
        )
    )

    return candidates[-1]


go_lib = latest_library(
    available_libs,
    "GO_Biological_Process"
)

reactome_lib = latest_library(
    available_libs,
    "Reactome"
)

hallmark_lib = latest_library(
    available_libs,
    "Hallmark"
)


selected_libs = [
    x for x in [
        go_lib,
        reactome_lib,
        hallmark_lib
    ]
    if x is not None
]


print(
    "Selected libraries:"
)

for x in selected_libs:
    print("  ", x)


# ============================================================
# 2. Download gene-set definitions
# ============================================================

gene_set_libraries = {}


for lib in selected_libs:

    print(
        "\nLoading:",
        lib
    )

    gene_set_libraries[
        lib
    ] = gp.get_library(
        name=lib,
        organism="Human"
    )


# ============================================================
# 3. Background genes
#
# IMPORTANT:
# background = genes actually tested,
# separately for MAC and FIB.
# ============================================================

backgrounds = {}


for celltype in [
    "MAC",
    "FIB"
]:

    bg = set(
        gene_results.loc[
            gene_results[
                "celltype"
            ]
            == celltype,
            "gene"
        ]
        .astype(str)
    )

    backgrounds[
        celltype
    ] = bg


    print(
        "\n",
        celltype,
        "background genes =",
        len(bg)
    )


# ============================================================
# 4. Local hypergeometric ORA
#
# We do NOT use whole-genome background.
# ============================================================

MIN_OVERLAP = 2

MIN_SET_SIZE_IN_PANEL = 3

MAX_SET_SIZE_IN_PANEL = 250


rows = []


for celltype in [
    "MAC",
    "FIB"
]:

    background = (
        backgrounds[
            celltype
        ]
    )

    M = len(
        background
    )


    modules = sorted(
        module_df.loc[
            module_df[
                "celltype"
            ]
            == celltype,
            "module"
        ].unique()
    )


    for module in modules:

        module_genes = set(
            module_df.loc[
                (
                    module_df[
                        "celltype"
                    ]
                    == celltype
                )
                &
                (
                    module_df[
                        "module"
                    ]
                    == module
                ),
                "gene"
            ]
            .astype(str)
        )


        # restrict query to tested universe
        module_genes = (
            module_genes
            &
            background
        )


        N = len(
            module_genes
        )


        print(
            f"\n{celltype} module {module}: "
            f"{N} genes"
        )


        for library_name, gene_sets in (
            gene_set_libraries.items()
        ):


            local_rows = []


            for term, pathway_genes in (
                gene_sets.items()
            ):

                pathway_in_panel = (
                    set(
                        str(x)
                        for x
                        in pathway_genes
                    )
                    &
                    background
                )


                K = len(
                    pathway_in_panel
                )


                if (
                    K
                    <
                    MIN_SET_SIZE_IN_PANEL
                    or
                    K
                    >
                    MAX_SET_SIZE_IN_PANEL
                ):

                    continue


                overlap = (
                    module_genes
                    &
                    pathway_in_panel
                )


                k = len(
                    overlap
                )


                if (
                    k
                    <
                    MIN_OVERLAP
                ):

                    continue


                # --------------------------
                # Hypergeometric test
                # --------------------------

                pvalue = hypergeom.sf(
                    k - 1,
                    M,
                    K,
                    N
                )


                # --------------------------
                # Fold enrichment
                # --------------------------

                observed_fraction = (
                    k / N
                )


                expected_fraction = (
                    K / M
                )


                fold_enrichment = (
                    observed_fraction
                    /
                    expected_fraction
                )


                local_rows.append({
                    "celltype":
                        celltype,

                    "module":
                        module,

                    "library":
                        library_name,

                    "term":
                        term,

                    "module_size":
                        N,

                    "pathway_size_in_panel":
                        K,

                    "overlap_n":
                        k,

                    "overlap_genes":
                        ";".join(
                            sorted(
                                overlap
                            )
                        ),

                    "fold_enrichment":
                        fold_enrichment,

                    "pvalue":
                        pvalue
                })


            # --------------------------------
            # BH inside module × library
            # --------------------------------

            if len(
                local_rows
            ) > 0:

                temp = pd.DataFrame(
                    local_rows
                )


                temp[
                    "FDR_within_library"
                ] = multipletests(
                    temp[
                        "pvalue"
                    ],
                    method="fdr_bh"
                )[1]


                rows.extend(
                    temp.to_dict(
                        orient="records"
                    )
                )


# ============================================================
# 5. Combined results
# ============================================================

enrich = pd.DataFrame(
    rows
)


if len(
    enrich
) == 0:

    raise ValueError(
        "No pathway terms had "
        ">=2 genes overlapping."
    )


# ============================================================
# 6. Additional conservative FDR
# across ALL libraries within each module
# ============================================================

enrich[
    "FDR_module_all_libraries"
] = (
    enrich
    .groupby(
        [
            "celltype",
            "module"
        ],
        observed=True
    )[
        "pvalue"
    ]
    .transform(
        lambda x:
        multipletests(
            x,
            method="fdr_bh"
        )[1]
    )
)


enrich = (
    enrich
    .sort_values(
        [
            "celltype",
            "module",
            "FDR_module_all_libraries",
            "pvalue",
            "fold_enrichment"
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False
        ]
    )
)


# ============================================================
# 7. Top pathway terms per module
# ============================================================

top_terms = (
    enrich
    .groupby(
        [
            "celltype",
            "module"
        ],
        observed=True
    )
    .head(
        10
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n================================"
)

print(
    "TOP PANEL-AWARE PATHWAY TERMS"
)

print(
    "================================"
)


display(
    top_terms[
        [
            "celltype",
            "module",
            "library",
            "term",
            "overlap_n",
            "overlap_genes",
            "fold_enrichment",
            "pvalue",
            "FDR_within_library",
            "FDR_module_all_libraries"
        ]
    ]
)


# ============================================================
# 8. Significant enrichment summary
# ============================================================

sig_summary = (
    enrich.assign(
        significant=
        enrich[
            "FDR_module_all_libraries"
        ] < 0.05
    )
    .groupby(
        [
            "celltype",
            "module"
        ],
        observed=True
    )
    .agg(
        tested_terms=(
            "term",
            "size"
        ),

        significant_terms=(
            "significant",
            "sum"
        )
    )
)


print(
    "\n================================"
)

print(
    "ENRICHMENT SUMMARY"
)

print(
    "================================"
)


display(
    sig_summary
)


# ============================================================
# 9. Save
# ============================================================

enrich.to_csv(
    base /
    "figure5_panel_aware_pathway_enrichment.csv",
    index=False
)


top_terms.to_csv(
    base /
    "figure5_panel_aware_top_pathways.csv",
    index=False
)


print(
    "\nSaved."
)

In [ ]:
# ============================================================
# FIGURE 5 — CLEAN MAIN-TEXT PATHWAY REPORT
#
# Main-text reporting rule:
#   conservative module-level FDR < 0.05
#   AND at least 3 overlapping genes
#
# No statistics are recomputed.
# ============================================================

from pathlib import Path
import pandas as pd


base = Path(str(PROJECT_DIR))


enrich = pd.read_csv(
    base /
    "figure5_panel_aware_pathway_enrichment.csv"
)


# ============================================================
# 1. Main-text-worthy enrichments
# ============================================================

main_enrich = enrich[
    (
        enrich[
            "FDR_module_all_libraries"
        ] < 0.05
    )
    &
    (
        enrich[
            "overlap_n"
        ] >= 3
    )
].copy()


main_enrich = (
    main_enrich
    .sort_values(
        [
            "celltype",
            "module",
            "FDR_module_all_libraries",
            "pvalue"
        ]
    )
)


print(
    "================================"
)

print(
    "MAIN-TEXT-WORTHY PATHWAYS"
)

print(
    "================================"
)


display(
    main_enrich[
        [
            "celltype",
            "module",
            "library",
            "term",
            "overlap_n",
            "overlap_genes",
            "fold_enrichment",
            "FDR_module_all_libraries"
        ]
    ]
)


# ============================================================
# 2. Number of robust pathway terms per module
# ============================================================

main_summary = (
    main_enrich
    .groupby(
        [
            "celltype",
            "module"
        ],
        observed=True
    )
    .agg(
        robust_terms=(
            "term",
            "size"
        ),

        best_FDR=(
            "FDR_module_all_libraries",
            "min"
        )
    )
)


print(
    "\n================================"
)

print(
    "ROBUST PATHWAY SUMMARY"
)

print(
    "================================"
)


display(
    main_summary
)


# ============================================================
# 3. Save
# ============================================================

main_enrich.to_csv(
    base /
    "figure5_maintext_pathway_enrichment.csv",
    index=False
)


print(
    "\nSaved."
)

In [ ]:
# ============================================================
# FIGURE 5 — COMPLETE FINAL VERSION
#
# A  MAC trajectory heatmap
# B  FIB trajectory heatmap
# C  MAC module 2 consensus
# D  FIB module 1 consensus
# E  FIB module 3 consensus
# F  panel-aware pathway enrichment
#
# Self-contained:
# reads saved Figure 5 files from disk
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from patsy import bs
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist


# ============================================================
# 0. Paths
# ============================================================

base = Path(str(PROJECT_DIR))

figdir = base / "figures"

figdir.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 1. Load saved data
# ============================================================

expr_df = pd.read_csv(
    base /
    "figure5_MAC_FIB_roi_mean_counts.csv"
)

module_df = pd.read_csv(
    base /
    "figure5_MAC_FIB_trajectory_modules.csv"
)

main_enrich = pd.read_csv(
    base /
    "figure5_maintext_pathway_enrichment.csv"
)


metadata_cols = [
    "roi_id",
    "celltype_l1",
    "n_cells",
    "Disease",
    "Patient",
    "PC1"
]


gene_cols = [
    c
    for c in expr_df.columns
    if c not in metadata_cols
]


print(
    "Expression:",
    expr_df.shape
)

print(
    "\nModules:"
)

print(
    module_df
    .groupby(
        [
            "celltype",
            "module"
        ],
        observed=True
    )
    .size()
)


# ============================================================
# 2. Recreate normalized log-expression
# ============================================================

X_raw = (
    expr_df[
        gene_cols
    ]
    .to_numpy(
        dtype=float
    )
)


row_sum = (
    X_raw.sum(
        axis=1
    )
)


X_norm = (
    X_raw
    /
    row_sum[:, None]
    *
    10000.0
)


X_log = np.log1p(
    X_norm
)


gene_index = {
    gene: i
    for i, gene
    in enumerate(
        gene_cols
    )
}


# ============================================================
# 3. Formal trajectory function
#
# anti-GBM minus LN
# patient-balanced
# patient-clustered
# ============================================================

def get_gene_delta(
    celltype,
    gene,
    n_grid=100
):

    if gene not in gene_index:
        return None


    mask = (
        expr_df[
            "celltype_l1"
        ]
        .astype(str)
        .to_numpy()
        ==
        celltype
    )


    meta = (
        expr_df.loc[
            mask,
            metadata_cols
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    y = (
        X_log[
            mask,
            gene_index[
                gene
            ]
        ]
    )


    d = meta[
        [
            "Disease",
            "Patient",
            "PC1"
        ]
    ].copy()


    d["y"] = y


    d["Disease"] = pd.Categorical(
        d["Disease"].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    # patient-balanced
    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )["Patient"]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )


    fit = smf.wls(
        (
            "y ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        ),
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    grid = np.linspace(
        d["PC1"].min(),
        d["PC1"].max(),
        n_grid
    )


    preds = {}


    for disease in [
        "SLE",
        "GBM"
    ]:

        newdata = pd.DataFrame({
            "PC1":
                grid,

            "Disease":
                pd.Categorical(
                    [disease]
                    *
                    len(grid),

                    categories=[
                        "SLE",
                        "GBM"
                    ]
                )
        })


        preds[
            disease
        ] = np.asarray(
            fit.predict(
                newdata
            )
        )


    delta = (
        preds["GBM"]
        -
        preds["SLE"]
    )


    return (
        grid,
        delta
    )


# ============================================================
# 4. Build trajectory matrices
# ============================================================

trajectory_data = {}


for celltype in [
    "MAC",
    "FIB"
]:

    genes = (
        module_df.loc[
            module_df[
                "celltype"
            ]
            == celltype,
            "gene"
        ]
        .astype(str)
        .tolist()
    )


    raw_curves = []

    z_curves = []

    kept_genes = []

    grid_saved = None


    print(
        "\nBuilding",
        celltype,
        "trajectories..."
    )


    for gene in genes:

        result = get_gene_delta(
            celltype,
            gene,
            n_grid=100
        )


        if result is None:
            continue


        grid, delta = result


        if not np.all(
            np.isfinite(
                delta
            )
        ):
            continue


        if np.std(
            delta
        ) < 1e-8:
            continue


        z = (
            delta
            -
            np.mean(
                delta
            )
        ) / np.std(
            delta
        )


        raw_curves.append(
            delta
        )

        z_curves.append(
            z
        )

        kept_genes.append(
            gene
        )

        grid_saved = grid


    raw_mat = np.vstack(
        raw_curves
    )

    z_mat = np.vstack(
        z_curves
    )


    # hierarchical order
    distance = pdist(
        z_mat,
        metric="correlation"
    )


    Z = linkage(
        distance,
        method="average"
    )


    order = leaves_list(
        Z
    )


    trajectory_data[
        celltype
    ] = {
        "genes":
            np.array(
                kept_genes
            ),

        "raw":
            raw_mat,

        "z":
            z_mat,

        "order":
            order,

        "grid":
            grid_saved
    }


# ============================================================
# 5. Module consensus helper
# ============================================================

def get_module_consensus(
    celltype,
    module
):

    genes = set(
        module_df.loc[
            (
                module_df[
                    "celltype"
                ]
                == celltype
            )
            &
            (
                module_df[
                    "module"
                ]
                == module
            ),
            "gene"
        ]
        .astype(str)
    )


    data = trajectory_data[
        celltype
    ]


    idx = [
        i
        for i, gene
        in enumerate(
            data["genes"]
        )
        if gene in genes
    ]


    curves = (
        data[
            "raw"
        ][
            idx,
            :
        ]
    )


    consensus = np.median(
        curves,
        axis=0
    )


    q25 = np.quantile(
        curves,
        0.25,
        axis=0
    )


    q75 = np.quantile(
        curves,
        0.75,
        axis=0
    )


    return (
        data[
            "grid"
        ],
        curves,
        consensus,
        q25,
        q75
    )


# ============================================================
# 6. Build Figure
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(
        16,
        9
    )
)


axA = axes[0, 0]
axB = axes[0, 1]
axC = axes[0, 2]

axD = axes[1, 0]
axE = axes[1, 1]
axF = axes[1, 2]


# ============================================================
# A/B. Heatmap helper
# ============================================================

def heatmap_panel(
    ax,
    celltype,
    panel
):

    data = trajectory_data[
        celltype
    ]


    ordered = (
        data[
            "z"
        ][
            data[
                "order"
            ],
            :
        ]
    )


    ordered_genes = (
        data[
            "genes"
        ][
            data[
                "order"
            ]
        ]
    )


    im = ax.imshow(
        ordered,
        aspect="auto",
        interpolation="nearest",
        cmap="coolwarm",
        vmin=-2.5,
        vmax=2.5
    )


    grid = data[
        "grid"
    ]


    xticks = np.linspace(
        0,
        len(grid) - 1,
        5
    )


    labels = np.linspace(
        grid.min(),
        grid.max(),
        5
    )


    ax.set_xticks(
        xticks
    )


    ax.set_xticklabels(
        [
            f"{x:.1f}"
            for x in labels
        ]
    )


    ax.set_xlabel(
        "Crescent progression (PC1)"
    )


    ax.set_ylabel(
        "Trajectory-divergent genes"
    )


    ax.set_title(
        (
            f"{panel}  {celltype} molecular trajectories\n"
            "anti-GBM − LN"
        ),
        loc="left",
        fontsize=11
    )


    # Label only a limited number of genes
    module_subset = (
        module_df[
            module_df[
                "celltype"
            ]
            == celltype
        ]
        .sort_values(
            "max_abs_delta",
            ascending=False
        )
        .head(
            12
        )
    )


    label_genes = set(
        module_subset[
            "gene"
        ]
        .astype(str)
    )


    yticks = []

    ylabels = []


    for i, gene in enumerate(
        ordered_genes
    ):

        if gene in label_genes:

            yticks.append(
                i
            )

            ylabels.append(
                gene
            )


    ax.set_yticks(
        yticks
    )


    ax.set_yticklabels(
        ylabels,
        fontsize=6.5
    )


    return im


imA = heatmap_panel(
    axA,
    "MAC",
    "A"
)


imB = heatmap_panel(
    axB,
    "FIB",
    "B"
)


# ============================================================
# C/D/E. Module trajectory helper
# ============================================================

def module_panel(
    ax,
    celltype,
    module,
    title,
    panel
):

    (
        grid,
        curves,
        consensus,
        q25,
        q75
    ) = get_module_consensus(
        celltype,
        module
    )


    # individual genes
    for curve in curves:

        ax.plot(
            grid,
            curve,
            linewidth=0.8,
            alpha=0.12
        )


    # interquartile range
    ax.fill_between(
        grid,
        q25,
        q75,
        alpha=0.15
    )


    # median consensus
    ax.plot(
        grid,
        consensus,
        linewidth=3,
        color="black"
    )


    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )


    ax.set_xlabel(
        "Crescent progression (PC1)"
    )


    ax.set_ylabel(
        "anti-GBM − LN\nlog-expression"
    )


    ax.set_title(
        (
            f"{panel}  {title}\n"
            f"n={len(curves)} genes"
        ),
        loc="left",
        fontsize=11
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )

    ax.spines[
        "right"
    ].set_visible(
        False
    )


# ============================================================
# C. MAC module 2
# ============================================================

module_panel(
    axC,
    "MAC",
    2,
    "MAC innate/complement-associated module",
    "C"
)


# ============================================================
# D. FIB module 1
# ============================================================

module_panel(
    axD,
    "FIB",
    1,
    "FIB wound-healing-associated module",
    "D"
)


# ============================================================
# E. FIB module 3
# ============================================================

module_panel(
    axE,
    "FIB",
    3,
    "FIB complement-associated module",
    "E"
)


# ============================================================
# F. Panel-aware enrichment dot plot
# ============================================================

ax = axF


# Keep six robust terms
plot_enrich = (
    main_enrich
    .copy()
)


# Short readable labels
term_map = {
    "Wound Healing, Spreading of Cells (GO:0044319)":
        "Wound healing",

    "Terminal Pathway of Complement":
        "Terminal complement",

    "Innate Immune System":
        "Innate immune system",

    "Neutrophil Degranulation":
        "Innate degranulation",

    "Complement Cascade":
        "Complement cascade",

    "Regulation of Complement Cascade":
        "Complement regulation"
}


plot_enrich[
    "display_term"
] = (
    plot_enrich[
        "term"
    ]
    .map(
        term_map
    )
    .fillna(
        plot_enrich[
            "term"
        ]
    )
)


plot_enrich[
    "module_label"
] = (
    plot_enrich[
        "celltype"
    ]
    +
    " M"
    +
    plot_enrich[
        "module"
    ].astype(str)
)


# Sort for plotting
plot_enrich = (
    plot_enrich
    .sort_values(
        [
            "celltype",
            "module",
            "FDR_module_all_libraries"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


y = np.arange(
    len(
        plot_enrich
    )
)


x = (
    -np.log10(
        plot_enrich[
            "FDR_module_all_libraries"
        ]
    )
)


sizes = (
    plot_enrich[
        "overlap_n"
    ]
    *
    45
)


sc = ax.scatter(
    x,
    y,
    s=sizes,
    c=plot_enrich[
        "fold_enrichment"
    ],
    cmap="viridis",
    edgecolor="black",
    linewidth=0.5
)


ax.set_yticks(
    y
)


ax.set_yticklabels(
    [
        (
            f"{row.module_label}: "
            f"{row.display_term}"
        )
        for row
        in plot_enrich.itertuples()
    ],
    fontsize=8
)


ax.set_xlabel(
    "-log10(panel-aware FDR)"
)


ax.set_title(
    "F  Panel-aware pathway enrichment",
    loc="left",
    fontsize=11
)


ax.spines[
    "top"
].set_visible(
    False
)


ax.spines[
    "right"
].set_visible(
    False
)


cbar = fig.colorbar(
    sc,
    ax=ax,
    fraction=0.05,
    pad=0.03
)


cbar.set_label(
    "Fold enrichment"
)


# ============================================================
# 7. Common heatmap colorbar
# ============================================================

cbar2 = fig.colorbar(
    imA,
    ax=[
        axA,
        axB
    ],
    fraction=0.02,
    pad=0.02
)


cbar2.set_label(
    "Within-gene standardized\nanti-GBM − LN difference"
)


# ============================================================
# 8. Final layout
# ============================================================

plt.tight_layout(
    w_pad=2.7,
    h_pad=3.0
)


# ============================================================
# 9. Save
# ============================================================

png_path = (
    figdir /
    "Figure5_COMPLETE_MAC_FIB_molecular_programs.png"
)


pdf_path = (
    figdir /
    "Figure5_COMPLETE_MAC_FIB_molecular_programs.pdf"
)


plt.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.savefig(
    pdf_path,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


print(
    "\nFigure 5 saved:"
)

print(
    png_path
)

print(
    pdf_path
)

In [ ]:
# ============================================================
# CRITICAL AUDIT — XENIUM SLIDE / DISEASE DISTRIBUTION
# ============================================================

from pathlib import Path
import anndata as ad
import pandas as pd

base = Path(str(PROJECT_DIR))

roi_ad = ad.read_h5ad(
    base / "roi_782_PC1_primary.h5ad"
)

meta = roi_ad.obs.copy()

meta.index = meta.index.astype(str)


# ============================================================
# 1. Extract Xenium slide ID
# ============================================================

if "Biopsy_ID" in meta.columns:

    meta["Slide"] = (
        meta["Biopsy_ID"]
        .astype(str)
        .str.extract(r"(\d{7})", expand=False)
    )

else:

    meta["Slide"] = (
        meta.index.to_series()
        .str.extract(r"(\d{7})", expand=False)
        .values
    )


print("Slides:")
print(
    sorted(
        meta["Slide"]
        .dropna()
        .unique()
    )
)


# ============================================================
# 2. ROI distribution across Disease × Slide
# ============================================================

print("\n==============================")
print("ROI COUNTS: DISEASE × SLIDE")
print("==============================")

roi_table = pd.crosstab(
    meta["Disease"],
    meta["Slide"]
)

display(roi_table)


# ============================================================
# 3. Patient distribution across Disease × Slide
# ============================================================

patient_table = (
    meta[
        [
            "Disease",
            "Slide",
            "Patient_Sample_ID"
        ]
    ]
    .drop_duplicates()
    .groupby(
        ["Disease", "Slide"],
        observed=True
    )["Patient_Sample_ID"]
    .nunique()
    .unstack(fill_value=0)
)

print("\n==============================")
print("PATIENT COUNTS: DISEASE × SLIDE")
print("==============================")

display(patient_table)


# ============================================================
# 4. Specifically LN vs anti-GBM common PC1 support
# ============================================================

pc_ranges = (
    meta[
        meta["Disease"].isin(
            ["ANCA", "SLE", "GBM"]
        )
    ]
    .groupby(
        "Disease",
        observed=True
    )["PC1_crescent"]
    .agg(["min", "max"])
)

pc_low = pc_ranges["min"].max()
pc_high = pc_ranges["max"].min()

common = meta[
    meta["Disease"].isin(["SLE", "GBM"])
    &
    meta["PC1_crescent"].between(
        pc_low,
        pc_high
    )
].copy()


print("\n==============================")
print("COMMON-SUPPORT ROI COUNTS")
print("==============================")

display(
    pd.crosstab(
        common["Disease"],
        common["Slide"]
    )
)


print("\n==============================")
print("COMMON-SUPPORT PATIENT COUNTS")
print("==============================")

display(
    common[
        [
            "Disease",
            "Slide",
            "Patient_Sample_ID"
        ]
    ]
    .drop_duplicates()
    .groupby(
        ["Disease", "Slide"],
        observed=True
    )["Patient_Sample_ID"]
    .nunique()
    .unstack(fill_value=0)
)

In [ ]:
# ============================================================
# CRITICAL SLIDE-ADJUSTED SENSITIVITY
#
# Outcome:
# MAC-anchored FIB neighborhood enrichment
#
# Model:
# outcome ~ spline(PC1) * Disease + Slide
#
# patient-balanced
# patient-clustered
#
# Two analyses:
# A. all common-support LN/GBM ROIs
# B. only slides containing BOTH LN and anti-GBM
# ============================================================

from pathlib import Path

import anndata as ad
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

from patsy import bs


# ============================================================
# 0. Load data
# ============================================================

base = Path(str(PROJECT_DIR))

roi_ad = ad.read_h5ad(
    base / "roi_782_PC1_primary.h5ad"
)

spatial = pd.read_csv(
    base / "figure4_driver_neighbor_data_smoothed.csv",
    index_col=0
)

spatial.index = spatial.index.astype(str)

meta = roi_ad.obs.copy()
meta.index = meta.index.astype(str)


# ============================================================
# 1. Reconstruct slide ID
# ============================================================

if "Biopsy_ID" in meta.columns:

    meta["Slide"] = (
        meta["Biopsy_ID"]
        .astype(str)
        .str.extract(
            r"(\d{7})",
            expand=False
        )
    )

else:

    meta["Slide"] = (
        meta.index.to_series()
        .str.extract(
            r"(\d{7})",
            expand=False
        )
        .values
    )


spatial["Slide"] = (
    meta.loc[
        spatial.index,
        "Slide"
    ]
    .astype(str)
)


# ============================================================
# 2. Keep LN / anti-GBM
# ============================================================

d0 = spatial[
    spatial["Disease"].isin(
        ["SLE", "GBM"]
    )
].dropna(
    subset=[
        "MAC_to_FIB",
        "PC1",
        "Patient",
        "Slide"
    ]
).copy()


print(
    "All common-support ROIs:",
    len(d0)
)

print(
    "Patients:",
    d0["Patient"].nunique()
)

print(
    "GBM patients:",
    d0.loc[
        d0["Disease"] == "GBM",
        "Patient"
    ].nunique()
)


# ============================================================
# 3. Find slides containing BOTH diseases
# ============================================================

slide_disease = (
    d0.groupby(
        "Slide",
        observed=True
    )["Disease"]
    .agg(
        lambda x:
        set(
            x.astype(str)
        )
    )
)


overlap_slides = [
    slide
    for slide, diseases
    in slide_disease.items()
    if (
        "SLE" in diseases
        and
        "GBM" in diseases
    )
]


print(
    "\nOverlap slides:"
)

print(
    overlap_slides
)


print(
    "\nNumber of overlap slides:",
    len(overlap_slides)
)


# ============================================================
# 4. Formal interaction testing function
# ============================================================

def slide_adjusted_test(
    data,
    outcome="MAC_to_FIB"
):

    d = data.dropna(
        subset=[
            outcome,
            "Disease",
            "Patient",
            "PC1",
            "Slide"
        ]
    ).copy()


    d["Disease"] = pd.Categorical(
        d["Disease"].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    d["Slide"] = pd.Categorical(
        d["Slide"].astype(str)
    )


    # --------------------------------
    # patient-balanced weights
    # --------------------------------

    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )["Patient"]
        .transform("size")
    )


    d["patient_weight"] = (
        1.0 /
        n_roi
    )


    # --------------------------------
    # slide-adjusted formal model
    # --------------------------------

    fit = smf.wls(
        (
            f"{outcome} ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease) "
            "+ C(Slide)"
        ),
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d["Patient"]
        }
    )


    # --------------------------------
    # joint test of Disease × PC1
    # --------------------------------

    interaction_terms = [
        term
        for term
        in fit.params.index
        if ":" in term
        and "C(Disease)" in term
        and "bs(PC1" in term
    ]


    R = np.zeros(
        (
            len(interaction_terms),
            len(fit.params)
        )
    )


    for i, term in enumerate(
        interaction_terms
    ):

        R[
            i,
            fit.params.index.get_loc(
                term
            )
        ] = 1


    wt = fit.wald_test(
        R,
        scalar=True
    )


    return {
        "pvalue":
            float(
                wt.pvalue
            ),

        "matrix_rank":
            int(
                np.linalg.matrix_rank(
                    fit.model.exog
                )
            ),

        "n_columns":
            int(
                fit.model.exog.shape[1]
            ),

        "n_ROI":
            len(d),

        "n_patients":
            d[
                "Patient"
            ].nunique(),

        "n_GBM_patients":
            d.loc[
                d[
                    "Disease"
                ].astype(str)
                == "GBM",
                "Patient"
            ].nunique(),

        "n_slides":
            d[
                "Slide"
            ].nunique()
    }


# ============================================================
# 5. Analysis A:
# all common-support ROIs + slide adjustment
# ============================================================

result_all = slide_adjusted_test(
    d0
)


# ============================================================
# 6. Analysis B:
# overlap slides only + slide adjustment
# ============================================================

d_overlap = d0[
    d0["Slide"].isin(
        overlap_slides
    )
].copy()


result_overlap = slide_adjusted_test(
    d_overlap
)


# ============================================================
# 7. Summary
# ============================================================

slide_results = pd.DataFrame(
    [
        {
            "analysis":
                "All common-support ROIs + Slide",

            **result_all
        },

        {
            "analysis":
                "Overlap slides only + Slide",

            **result_overlap
        }
    ]
)


print(
    "\n======================================"
)

print(
    "SLIDE-ADJUSTED MAC–FIB RESULTS"
)

print(
    "======================================"
)


display(
    slide_results
)


print(
    "\nOverlap-only disease counts:"
)

print(
    d_overlap[
        "Disease"
    ].value_counts()
)


print(
    "\nOverlap-only patient counts:"
)

print(
    d_overlap.groupby(
        "Disease",
        observed=True
    )["Patient"].nunique()
)


# ============================================================
# 8. Rank check
# ============================================================

print(
    "\nNon-full-rank models:"
)

display(
    slide_results[
        slide_results[
            "matrix_rank"
        ]
        !=
        slide_results[
            "n_columns"
        ]
    ]
)


# ============================================================
# 9. Save
# ============================================================

slide_results.to_csv(
    base /
    "figure4_MAC_FIB_slide_adjusted_sensitivity.csv",
    index=False
)


print(
    "\nSaved."
)

In [ ]:
# ============================================================
# DIAGNOSTIC:
# Did slide adjustment CHANGE the MAC-FIB trajectory,
# or mainly increase uncertainty?
#
# Compare:
# 1. Unadjusted
# 2. All ROIs + Slide
# 3. Overlap slides only + Slide
# ============================================================

from pathlib import Path

import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from patsy import bs


# ============================================================
# 0. Load
# ============================================================

base = Path(str(PROJECT_DIR))

roi_ad = ad.read_h5ad(
    base / "roi_782_PC1_primary.h5ad"
)

spatial = pd.read_csv(
    base / "figure4_driver_neighbor_data_smoothed.csv",
    index_col=0
)

spatial.index = spatial.index.astype(str)

meta = roi_ad.obs.copy()
meta.index = meta.index.astype(str)


# ============================================================
# 1. Slide ID
# ============================================================

if "Biopsy_ID" in meta.columns:

    meta["Slide"] = (
        meta["Biopsy_ID"]
        .astype(str)
        .str.extract(
            r"(\d{7})",
            expand=False
        )
    )

else:

    meta["Slide"] = (
        meta.index.to_series()
        .str.extract(
            r"(\d{7})",
            expand=False
        )
        .values
    )


spatial["Slide"] = (
    meta.loc[
        spatial.index,
        "Slide"
    ]
    .astype(str)
)


# ============================================================
# 2. LN vs anti-GBM
# ============================================================

d0 = spatial[
    spatial["Disease"].isin(
        ["SLE", "GBM"]
    )
].dropna(
    subset=[
        "MAC_to_FIB",
        "PC1",
        "Patient",
        "Slide"
    ]
).copy()


# overlap slides
slide_disease = (
    d0.groupby(
        "Slide",
        observed=True
    )["Disease"]
    .agg(
        lambda x: set(
            x.astype(str)
        )
    )
)

overlap_slides = [
    slide
    for slide, diseases
    in slide_disease.items()
    if (
        "SLE" in diseases
        and
        "GBM" in diseases
    )
]

d_overlap = d0[
    d0["Slide"].isin(
        overlap_slides
    )
].copy()


# ============================================================
# 3. Fit helper
# ============================================================

def fit_model(
    data,
    add_slide=False
):

    d = data.copy()

    d["Disease"] = pd.Categorical(
        d["Disease"].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )

    if add_slide:

        d["Slide"] = pd.Categorical(
            d["Slide"].astype(str)
        )


    # patient-balanced weights
    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )["Patient"]
        .transform("size")
    )

    d["patient_weight"] = (
        1.0 / n_roi
    )


    if add_slide:

        formula = (
            "MAC_to_FIB ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease) "
            "+ C(Slide)"
        )

    else:

        formula = (
            "MAC_to_FIB ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        )


    fit = smf.wls(
        formula,
        data=d,
        weights=d["patient_weight"]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": d["Patient"]
        }
    )


    # joint interaction P
    interaction_terms = [
        term
        for term in fit.params.index
        if ":" in term
        and "C(Disease)" in term
        and "bs(PC1" in term
    ]

    R = np.zeros(
        (
            len(interaction_terms),
            len(fit.params)
        )
    )

    for i, term in enumerate(
        interaction_terms
    ):

        R[
            i,
            fit.params.index.get_loc(term)
        ] = 1


    p = float(
        fit.wald_test(
            R,
            scalar=True
        ).pvalue
    )


    return d, fit, p


# ============================================================
# 4. Three models
# ============================================================

d_unadj, fit_unadj, p_unadj = fit_model(
    d0,
    add_slide=False
)

d_slide, fit_slide, p_slide = fit_model(
    d0,
    add_slide=True
)

d_overlap2, fit_overlap, p_overlap = fit_model(
    d_overlap,
    add_slide=True
)


# ============================================================
# 5. Common prediction range across all three analyses
# ============================================================

pc_low = max(
    d_unadj["PC1"].min(),
    d_slide["PC1"].min(),
    d_overlap2["PC1"].min()
)

pc_high = min(
    d_unadj["PC1"].max(),
    d_slide["PC1"].max(),
    d_overlap2["PC1"].max()
)

grid = np.linspace(
    pc_low,
    pc_high,
    150
)


# ============================================================
# 6. Disease-difference curve helper
#
# GBM - LN
#
# For slide-adjusted models:
# predict both diseases at every slide, then average equally
# across slides.
# ============================================================

def disease_difference_curve(
    fit,
    data,
    add_slide=False
):

    if not add_slide:

        pred = {}

        for disease in [
            "SLE",
            "GBM"
        ]:

            newdata = pd.DataFrame({
                "PC1": grid,

                "Disease":
                    pd.Categorical(
                        [disease] * len(grid),
                        categories=[
                            "SLE",
                            "GBM"
                        ]
                    )
            })

            pred[disease] = np.asarray(
                fit.predict(
                    newdata
                )
            )

        return (
            pred["GBM"]
            -
            pred["SLE"]
        )


    slides = sorted(
        data["Slide"]
        .astype(str)
        .unique()
    )

    disease_preds = {
        "SLE": [],
        "GBM": []
    }


    for slide in slides:

        for disease in [
            "SLE",
            "GBM"
        ]:

            newdata = pd.DataFrame({
                "PC1": grid,

                "Disease":
                    pd.Categorical(
                        [disease] * len(grid),
                        categories=[
                            "SLE",
                            "GBM"
                        ]
                    ),

                "Slide":
                    pd.Categorical(
                        [slide] * len(grid),
                        categories=
                            fit.model.data.frame[
                                "Slide"
                            ].cat.categories
                    )
            })


            disease_preds[
                disease
            ].append(
                np.asarray(
                    fit.predict(
                        newdata
                    )
                )
            )


    mean_SLE = np.mean(
        np.vstack(
            disease_preds["SLE"]
        ),
        axis=0
    )

    mean_GBM = np.mean(
        np.vstack(
            disease_preds["GBM"]
        ),
        axis=0
    )


    return (
        mean_GBM
        -
        mean_SLE
    )


# ============================================================
# 7. Get curves
# ============================================================

curve_unadj = disease_difference_curve(
    fit_unadj,
    d_unadj,
    add_slide=False
)

curve_slide = disease_difference_curve(
    fit_slide,
    d_slide,
    add_slide=True
)

curve_overlap = disease_difference_curve(
    fit_overlap,
    d_overlap2,
    add_slide=True
)


# ============================================================
# 8. Quantify similarity
# ============================================================

def curve_metrics(
    reference,
    test
):

    corr = np.corrcoef(
        reference,
        test
    )[0, 1]

    rmse = np.sqrt(
        np.mean(
            (
                reference
                -
                test
            ) ** 2
        )
    )

    amplitude_ref = (
        np.max(reference)
        -
        np.min(reference)
    )

    amplitude_test = (
        np.max(test)
        -
        np.min(test)
    )

    return {
        "curve_correlation":
            corr,

        "RMSE":
            rmse,

        "reference_amplitude":
            amplitude_ref,

        "test_amplitude":
            amplitude_test
    }


metrics_slide = curve_metrics(
    curve_unadj,
    curve_slide
)

metrics_overlap = curve_metrics(
    curve_unadj,
    curve_overlap
)


summary = pd.DataFrame([
    {
        "model":
            "Unadjusted",

        "interaction_p":
            p_unadj,

        "curve_correlation_vs_unadjusted":
            1.0,

        "curve_amplitude":
            np.max(curve_unadj)
            -
            np.min(curve_unadj),

        "n_ROI":
            len(d_unadj),

        "n_patients":
            d_unadj["Patient"].nunique()
    },

    {
        "model":
            "All + Slide",

        "interaction_p":
            p_slide,

        "curve_correlation_vs_unadjusted":
            metrics_slide[
                "curve_correlation"
            ],

        "curve_amplitude":
            np.max(curve_slide)
            -
            np.min(curve_slide),

        "n_ROI":
            len(d_slide),

        "n_patients":
            d_slide["Patient"].nunique()
    },

    {
        "model":
            "Overlap slides + Slide",

        "interaction_p":
            p_overlap,

        "curve_correlation_vs_unadjusted":
            metrics_overlap[
                "curve_correlation"
            ],

        "curve_amplitude":
            np.max(curve_overlap)
            -
            np.min(curve_overlap),

        "n_ROI":
            len(d_overlap2),

        "n_patients":
            d_overlap2["Patient"].nunique()
    }
])


print(
    "===================================="
)

print(
    "SLIDE DIAGNOSTIC SUMMARY"
)

print(
    "===================================="
)

display(
    summary
)


# ============================================================
# 9. Plot difference curves
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.2, 5.4)
)


ax.plot(
    grid,
    curve_unadj,
    linewidth=2.8,
    label=f"Unadjusted, P={p_unadj:.3g}"
)


ax.plot(
    grid,
    curve_slide,
    linewidth=2.2,
    label=f"+ Slide, P={p_slide:.3g}"
)


ax.plot(
    grid,
    curve_overlap,
    linewidth=2.2,
    label=f"Overlap slides + Slide, P={p_overlap:.3g}"
)


ax.axhline(
    0,
    linestyle="--",
    linewidth=1
)


ax.set_xlabel(
    "Crescent progression (PC1)"
)

ax.set_ylabel(
    "Predicted anti-GBM − LN\nMAC–FIB neighborhood enrichment"
)

ax.set_title(
    "Effect of slide adjustment on MAC–FIB trajectory",
    loc="left"
)

ax.legend(
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FIGURE 5 — CRITICAL SLIDE-ADJUSTED MODULE AUDIT
#
# Frozen modules only:
#
# MAC M2 = innate/complement-associated
# FIB M1 = wound-healing-associated
# FIB M3 = complement-related
#
# Module score:
#   1) log-normalized expression
#   2) gene-wise z-score within cell type
#   3) mean z-score across module genes
#
# Models:
# A. all common-support units, unadjusted
# B. all common-support units + Slide
# C. disease-overlapping slides only + Slide
#
# patient-balanced
# patient-clustered
#
# IMPORTANT:
# This is a sensitivity analysis, not independent validation.
# ============================================================

from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from patsy import bs
from statsmodels.stats.multitest import multipletests


# ============================================================
# 0. Paths
# ============================================================

base = Path(
    str(PROJECT_DIR)
)

figdir = (
    base /
    "figures"
)

figdir.mkdir(
    parents=True,
    exist_ok=True
)


expr_path = (
    base /
    "figure5_MAC_FIB_roi_mean_counts.csv"
)

module_path = (
    base /
    "figure5_MAC_FIB_trajectory_modules.csv"
)

roi_path = (
    base /
    "roi_782_PC1_primary.h5ad"
)


# ============================================================
# 1. Load small files
# ============================================================

expr_df = pd.read_csv(
    expr_path
)

module_df = pd.read_csv(
    module_path
)

roi_ad = ad.read_h5ad(
    roi_path
)


roi_meta = (
    roi_ad.obs.copy()
)

roi_meta.index = (
    roi_meta.index.astype(str)
)


print(
    "Expression table:",
    expr_df.shape
)

print(
    "\nModule table:",
    module_df.shape
)


# ============================================================
# 2. Reconstruct Slide ID
# ============================================================

if "Biopsy_ID" in roi_meta.columns:

    roi_meta[
        "Slide"
    ] = (
        roi_meta[
            "Biopsy_ID"
        ]
        .astype(str)
        .str.extract(
            r"(\d{7})",
            expand=False
        )
    )

else:

    roi_meta[
        "Slide"
    ] = (
        roi_meta
        .index
        .to_series()
        .str.extract(
            r"(\d{7})",
            expand=False
        )
        .values
    )


expr_df[
    "roi_id"
] = (
    expr_df[
        "roi_id"
    ].astype(str)
)


expr_df[
    "Slide"
] = (
    expr_df[
        "roi_id"
    ].map(
        roi_meta[
            "Slide"
        ]
    )
)


if (
    expr_df[
        "Slide"
    ].isna().any()
):

    raise ValueError(
        "Some ROI units could not "
        "be mapped to a Xenium slide."
    )


print(
    "\nSlides:"
)

print(
    sorted(
        expr_df[
            "Slide"
        ]
        .astype(str)
        .unique()
    )
)


# ============================================================
# 3. Normalize expression exactly as before
# ============================================================

metadata_cols = [
    "roi_id",
    "celltype_l1",
    "n_cells",
    "Disease",
    "Patient",
    "PC1",
    "Slide"
]


gene_cols = [
    c
    for c
    in expr_df.columns
    if c not in metadata_cols
]


X_raw = (
    expr_df[
        gene_cols
    ]
    .to_numpy(
        dtype=float
    )
)


row_sum = (
    X_raw.sum(
        axis=1
    )
)


if (
    row_sum <= 0
).any():

    raise ValueError(
        "Found ROI×celltype unit "
        "with zero total expression."
    )


X_norm = (
    X_raw
    /
    row_sum[
        :,
        None
    ]
    *
    10000.0
)


X_log = np.log1p(
    X_norm
)


gene_index = {
    gene: i
    for i, gene
    in enumerate(
        gene_cols
    )
}


# ============================================================
# 4. FROZEN modules
# ============================================================

modules_to_audit = [
    {
        "celltype":
            "MAC",

        "module":
            2,

        "label":
            "MAC M2",

        "title":
            "MAC innate/complement-associated"
    },

    {
        "celltype":
            "FIB",

        "module":
            1,

        "label":
            "FIB M1",

        "title":
            "FIB wound-healing-associated"
    },

    {
        "celltype":
            "FIB",

        "module":
            3,

        "label":
            "FIB M3",

        "title":
            "FIB complement-related"
    }
]


# ============================================================
# 5. Build module scores
#
# For every module:
#
# - select ROI×celltype units
# - z-score EACH gene across those ROI units
# - average z-scores across genes
#
# This prevents highly expressed genes
# from dominating the module score.
# ============================================================

score_tables = []


for info in modules_to_audit:

    celltype = (
        info[
            "celltype"
        ]
    )

    module = (
        info[
            "module"
        ]
    )


    genes = (
        module_df.loc[
            (
                module_df[
                    "celltype"
                ]
                == celltype
            )
            &
            (
                module_df[
                    "module"
                ]
                == module
            ),
            "gene"
        ]
        .astype(str)
        .tolist()
    )


    genes = [
        g
        for g in genes
        if g in gene_index
    ]


    print(
        "\n",
        info[
            "label"
        ],
        ":",
        len(genes),
        "genes"
    )


    print(
        genes
    )


    mask = (
        expr_df[
            "celltype_l1"
        ]
        .astype(str)
        .to_numpy()
        ==
        celltype
    )


    meta_ct = (
        expr_df.loc[
            mask,
            [
                "roi_id",
                "Disease",
                "Patient",
                "PC1",
                "Slide"
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    positions = [
        gene_index[
            g
        ]
        for g in genes
    ]


    X_module = (
        X_log[
            mask,
            :
        ][
            :,
            positions
        ]
    )


    # --------------------------------
    # Gene-wise standardization
    # pooled across LN + anti-GBM
    #
    # Does not use PC1 labels.
    # --------------------------------

    means = np.mean(
        X_module,
        axis=0
    )


    sds = np.std(
        X_module,
        axis=0,
        ddof=0
    )


    usable = (
        sds > 1e-8
    )


    if (
        usable.sum()
        <
        2
    ):

        raise ValueError(
            f"{info['label']} has "
            "too few variable genes."
        )


    Z = (
        X_module[
            :,
            usable
        ]
        -
        means[
            usable
        ]
    ) / sds[
        usable
    ]


    module_score = (
        np.mean(
            Z,
            axis=1
        )
    )


    meta_ct[
        "module_score"
    ] = (
        module_score
    )


    meta_ct[
        "celltype"
    ] = (
        celltype
    )


    meta_ct[
        "module"
    ] = (
        module
    )


    meta_ct[
        "module_label"
    ] = (
        info[
            "label"
        ]
    )


    meta_ct[
        "module_title"
    ] = (
        info[
            "title"
        ]
    )


    meta_ct[
        "n_module_genes"
    ] = (
        int(
            usable.sum()
        )
    )


    score_tables.append(
        meta_ct
    )


module_scores = pd.concat(
    score_tables,
    ignore_index=True
)


# ============================================================
# 6. Coverage audit
# ============================================================

print(
    "\n===================================="
)

print(
    "MODULE SCORE COVERAGE"
)

print(
    "===================================="
)


coverage = (
    module_scores
    .groupby(
        [
            "module_label",
            "Disease"
        ],
        observed=True
    )
    .agg(
        n_ROI=(
            "roi_id",
            "size"
        ),

        n_patients=(
            "Patient",
            "nunique"
        ),

        n_slides=(
            "Slide",
            "nunique"
        )
    )
)


display(
    coverage
)


# ============================================================
# 7. Model helper
# ============================================================

def fit_module_model(
    data,
    add_slide=False
):

    d = data.dropna(
        subset=[
            "module_score",
            "Disease",
            "Patient",
            "PC1",
            "Slide"
        ]
    ).copy()


    d[
        "Disease"
    ] = pd.Categorical(
        d[
            "Disease"
        ].astype(str),
        categories=[
            "SLE",
            "GBM"
        ]
    )


    if add_slide:

        d[
            "Slide"
        ] = pd.Categorical(
            d[
                "Slide"
            ].astype(str)
        )


    # --------------------------------
    # Patient-balanced weights
    # --------------------------------

    n_roi = (
        d.groupby(
            "Patient",
            observed=True
        )[
            "Patient"
        ]
        .transform(
            "size"
        )
    )


    d[
        "patient_weight"
    ] = (
        1.0 /
        n_roi
    )


    # --------------------------------
    # Formal model
    # --------------------------------

    if add_slide:

        formula = (
            "module_score ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease) "
            "+ C(Slide)"
        )

    else:

        formula = (
            "module_score ~ "
            "bs(PC1, df=3, degree=3, "
            "include_intercept=False) "
            "* C(Disease)"
        )


    fit = smf.wls(
        formula,
        data=d,
        weights=d[
            "patient_weight"
        ]
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                d[
                    "Patient"
                ]
        }
    )


    # --------------------------------
    # Joint Disease × PC1 test
    # --------------------------------

    interaction_terms = [
        term
        for term
        in fit.params.index
        if ":" in term
        and
        "C(Disease)"
        in term
        and
        "bs(PC1"
        in term
    ]


    R = np.zeros(
        (
            len(
                interaction_terms
            ),
            len(
                fit.params
            )
        )
    )


    for i, term in enumerate(
        interaction_terms
    ):

        R[
            i,
            fit.params.index
            .get_loc(
                term
            )
        ] = 1


    wt = (
        fit.wald_test(
            R,
            scalar=True
        )
    )


    return (
        d,
        fit,
        float(
            wt.pvalue
        )
    )


# ============================================================
# 8. Difference-curve prediction helper
#
# anti-GBM minus LN
#
# For +Slide models we average predictions
# equally across the slides present in that model.
# ============================================================

def predict_difference_curve(
    fit,
    data,
    grid,
    add_slide=False
):

    if not add_slide:

        preds = {}


        for disease in [
            "SLE",
            "GBM"
        ]:

            newdata = pd.DataFrame({
                "PC1":
                    grid,

                "Disease":
                    pd.Categorical(
                        [disease]
                        *
                        len(grid),

                        categories=[
                            "SLE",
                            "GBM"
                        ]
                    )
            })


            preds[
                disease
            ] = np.asarray(
                fit.predict(
                    newdata
                )
            )


        return (
            preds[
                "GBM"
            ]
            -
            preds[
                "SLE"
            ]
        )


    slide_categories = (
        data[
            "Slide"
        ]
        .cat
        .categories
    )


    slides = list(
        slide_categories
    )


    preds = {
        "SLE": [],
        "GBM": []
    }


    for slide in slides:

        for disease in [
            "SLE",
            "GBM"
        ]:

            newdata = pd.DataFrame({
                "PC1":
                    grid,

                "Disease":
                    pd.Categorical(
                        [disease]
                        *
                        len(grid),

                        categories=[
                            "SLE",
                            "GBM"
                        ]
                    ),

                "Slide":
                    pd.Categorical(
                        [slide]
                        *
                        len(grid),

                        categories=
                            slide_categories
                    )
            })


            preds[
                disease
            ].append(
                np.asarray(
                    fit.predict(
                        newdata
                    )
                )
            )


    mean_SLE = np.mean(
        np.vstack(
            preds[
                "SLE"
            ]
        ),
        axis=0
    )


    mean_GBM = np.mean(
        np.vstack(
            preds[
                "GBM"
            ]
        ),
        axis=0
    )


    return (
        mean_GBM
        -
        mean_SLE
    )


# ============================================================
# 9. Run the three models for each frozen module
# ============================================================

result_rows = []

curve_store = {}


for info in modules_to_audit:

    label = (
        info[
            "label"
        ]
    )


    d0 = (
        module_scores[
            module_scores[
                "module_label"
            ]
            == label
        ]
        .copy()
    )


    # --------------------------------
    # Determine disease-overlap slides
    # for THIS module
    # --------------------------------

    slide_disease = (
        d0.groupby(
            "Slide",
            observed=True
        )[
            "Disease"
        ]
        .agg(
            lambda x:
            set(
                x.astype(str)
            )
        )
    )


    overlap_slides = [
        slide
        for slide, diseases
        in slide_disease.items()
        if (
            "SLE"
            in diseases
            and
            "GBM"
            in diseases
        )
    ]


    d_overlap = d0[
        d0[
            "Slide"
        ].astype(str)
        .isin(
            [
                str(x)
                for x
                in overlap_slides
            ]
        )
    ].copy()


    print(
        "\n================================"
    )

    print(
        label
    )

    print(
        "================================"
    )


    print(
        "Overlap slides:",
        overlap_slides
    )


    # --------------------------------
    # Fit 3 specifications
    # --------------------------------

    d_unadj, fit_unadj, p_unadj = (
        fit_module_model(
            d0,
            add_slide=False
        )
    )


    d_slide, fit_slide, p_slide = (
        fit_module_model(
            d0,
            add_slide=True
        )
    )


    (
        d_overlap2,
        fit_overlap,
        p_overlap
    ) = fit_module_model(
        d_overlap,
        add_slide=True
    )


    # --------------------------------
    # Common PC1 grid
    # across the 3 models
    # --------------------------------

    pc_low = max(
        d_unadj[
            "PC1"
        ].min(),

        d_slide[
            "PC1"
        ].min(),

        d_overlap2[
            "PC1"
        ].min()
    )


    pc_high = min(
        d_unadj[
            "PC1"
        ].max(),

        d_slide[
            "PC1"
        ].max(),

        d_overlap2[
            "PC1"
        ].max()
    )


    grid = np.linspace(
        pc_low,
        pc_high,
        150
    )


    # --------------------------------
    # Difference curves
    # --------------------------------

    curve_unadj = (
        predict_difference_curve(
            fit_unadj,
            d_unadj,
            grid,
            add_slide=False
        )
    )


    curve_slide = (
        predict_difference_curve(
            fit_slide,
            d_slide,
            grid,
            add_slide=True
        )
    )


    curve_overlap = (
        predict_difference_curve(
            fit_overlap,
            d_overlap2,
            grid,
            add_slide=True
        )
    )


    # --------------------------------
    # Shape similarity
    # --------------------------------

    corr_slide = float(
        np.corrcoef(
            curve_unadj,
            curve_slide
        )[
            0,
            1
        ]
    )


    corr_overlap = float(
        np.corrcoef(
            curve_unadj,
            curve_overlap
        )[
            0,
            1
        ]
    )


    amp_unadj = float(
        np.max(
            curve_unadj
        )
        -
        np.min(
            curve_unadj
        )
    )


    amp_slide = float(
        np.max(
            curve_slide
        )
        -
        np.min(
            curve_slide
        )
    )


    amp_overlap = float(
        np.max(
            curve_overlap
        )
        -
        np.min(
            curve_overlap
        )
    )


    # --------------------------------
    # Save curves
    # --------------------------------

    curve_store[
        label
    ] = {
        "grid":
            grid,

        "Unadjusted":
            curve_unadj,

        "+ Slide":
            curve_slide,

        "Overlap slides + Slide":
            curve_overlap
    }


    # --------------------------------
    # Summary rows
    # --------------------------------

    for (
        model_name,
        d_model,
        fit_model,
        pvalue,
        corr,
        amp
    ) in [

        (
            "Unadjusted",
            d_unadj,
            fit_unadj,
            p_unadj,
            1.0,
            amp_unadj
        ),

        (
            "All + Slide",
            d_slide,
            fit_slide,
            p_slide,
            corr_slide,
            amp_slide
        ),

        (
            "Overlap slides + Slide",
            d_overlap2,
            fit_overlap,
            p_overlap,
            corr_overlap,
            amp_overlap
        )
    ]:

        result_rows.append({
            "module":
                label,

            "module_title":
                info[
                    "title"
                ],

            "model":
                model_name,

            "interaction_p":
                pvalue,

            "curve_correlation_vs_unadjusted":
                corr,

            "curve_amplitude":
                amp,

            "matrix_rank":
                int(
                    np.linalg.matrix_rank(
                        fit_model.model.exog
                    )
                ),

            "n_columns":
                int(
                    fit_model.model.exog.shape[
                        1
                    ]
                ),

            "n_ROI":
                len(
                    d_model
                ),

            "n_patients":
                d_model[
                    "Patient"
                ].nunique(),

            "n_GBM_patients":
                d_model.loc[
                    d_model[
                        "Disease"
                    ]
                    .astype(str)
                    == "GBM",
                    "Patient"
                ].nunique(),

            "n_slides":
                d_model[
                    "Slide"
                ].nunique()
        })


# ============================================================
# 10. Summary table
# ============================================================

module_slide_results = pd.DataFrame(
    result_rows
)


# ============================================================
# 11. BH across the 3 frozen modules
# within each sensitivity specification
#
# Again:
# descriptive sensitivity only,
# NOT independent confirmation.
# ============================================================

module_slide_results[
    "FDR_across_3_modules"
] = (
    module_slide_results
    .groupby(
        "model",
        observed=True
    )[
        "interaction_p"
    ]
    .transform(
        lambda x:
        multipletests(
            x,
            method="fdr_bh"
        )[1]
    )
)


print(
    "\n===================================="
)

print(
    "FIGURE 5 SLIDE-ADJUSTED MODULE SUMMARY"
)

print(
    "===================================="
)


display(
    module_slide_results[
        [
            "module",
            "model",
            "interaction_p",
            "FDR_across_3_modules",
            "curve_correlation_vs_unadjusted",
            "curve_amplitude",
            "matrix_rank",
            "n_columns",
            "n_ROI",
            "n_patients",
            "n_GBM_patients",
            "n_slides"
        ]
    ]
)


# ============================================================
# 12. Rank audit
# ============================================================

print(
    "\nNon-full-rank models:"
)

display(
    module_slide_results[
        module_slide_results[
            "matrix_rank"
        ]
        !=
        module_slide_results[
            "n_columns"
        ]
    ]
)


# ============================================================
# 13. Plot trajectory geometry
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(
        15,
        4.8
    )
)


for ax, info in zip(
    axes,
    modules_to_audit
):

    label = (
        info[
            "label"
        ]
    )


    curves = (
        curve_store[
            label
        ]
    )


    grid = (
        curves[
            "grid"
        ]
    )


    rows = (
        module_slide_results[
            module_slide_results[
                "module"
            ]
            == label
        ]
        .set_index(
            "model"
        )
    )


    ax.plot(
        grid,
        curves[
            "Unadjusted"
        ],
        linewidth=2.7,
        label=(
            "Unadjusted "
            f"P={rows.loc['Unadjusted', 'interaction_p']:.3g}"
        )
    )


    ax.plot(
        grid,
        curves[
            "+ Slide"
        ],
        linewidth=2.2,
        label=(
            "+ Slide "
            f"P={rows.loc['All + Slide', 'interaction_p']:.3g}"
        )
    )


    ax.plot(
        grid,
        curves[
            "Overlap slides + Slide"
        ],
        linewidth=2.2,
        label=(
            "Overlap + Slide "
            f"P={rows.loc['Overlap slides + Slide', 'interaction_p']:.3g}"
        )
    )


    ax.axhline(
        0,
        linestyle="--",
        linewidth=1
    )


    ax.set_xlabel(
        "Crescent progression (PC1)"
    )


    ax.set_ylabel(
        "Predicted anti-GBM − LN\nmodule score"
    )


    ax.set_title(
        info[
            "title"
        ],
        loc="left",
        fontsize=11
    )


    ax.legend(
        frameon=False,
        fontsize=8
    )


    ax.spines[
        "top"
    ].set_visible(
        False
    )


    ax.spines[
        "right"
    ].set_visible(
        False
    )


fig.subplots_adjust(
    left=0.07,
    right=0.98,
    bottom=0.16,
    top=0.88,
    wspace=0.32
)


slide_fig = (
    figdir /
    "Figure5_SLIDE_AUDIT_frozen_modules.png"
)


plt.savefig(
    slide_fig,
    dpi=500,
    bbox_inches="tight",
    facecolor="white",
    transparent=False
)


plt.show()


# ============================================================
# 14. Save tables
# ============================================================

module_scores.to_csv(
    base /
    "figure5_frozen_module_scores.csv",
    index=False
)


module_slide_results.to_csv(
    base /
    "figure5_slide_adjusted_module_sensitivity.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    slide_fig
)